# Programacion Funcional: Pipeline Integral End-to-End en Google Colab
## Monitoreo y Análisis de Sentimiento de Noticias Deportivas en Español con GDELT y PySpark

Este notebook ejecuta el flujo de datos completo y autónomo de extremo a extremo:
1. **Fase 1 (Extracción & ETL)**: Descarga multihilo en tiempo real desde los servidores de **GDELT 2.0 Global Knowledge Graph** (672 lotes, 100% de cobertura continua cada 15 min de los últimos 7 días).
2. **Fase 2 (Filtrado & Creación del Dataset)**: Detección estricta de idioma español (`srclc:spa`), filtrado de 10 disciplinas deportivas, deduplicación y almacenamiento en CSV plano.
3. **Fase 3 (Big Data con PySpark)**: Procesamiento distribuido en Apache Spark (tablas pivote, Window Functions analíticas, correlación léxica de sentimientos y polaridad).
4. **Fase 4 (Visualización)**: Generación interactiva de Matrices de Calor (*Heatmaps*), Gráficos de Concentración (*Donut Charts*) y Series Temporales diarias.

---
### Fase 0: Instalación y Configuración del Entorno

In [ ]:
!pip install -q pyspark

---
### Fase 1: Carga o Extracción del Dataset (GDELT GKG)
Puedes subir directamente `gdelt_deportes_semana_completa.csv` arrastrándolo a la sección **Archivos** (icono de carpeta a la izquierda en Colab) para omitir la descarga. Si no se detecta el archivo, se solicitará la subida o se descargará en vivo.

In [ ]:
import io
import csv
import json
import re
import time
import zipfile
import requests
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

# 1. Conexión HTTP persistente de alta velocidad
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=32, pool_maxsize=32, max_retries=2)
session.mount('http://', adapter)
session.mount('https://', adapter)

# 2. Diccionarios taxonómicos y reglas de detección de deportes
SPORTS_DOMAINS = {
    'marca.com', 'as.com', 'ole.com.ar', 'mundodeportivo.com', 'sport.es',
    'tycsports.com', 'depor.com', 'record.com.mx', 'mediotiempo.com',
    'elgrafico.com.ar', 'redgol.cl', 'futbolred.com', 'estadiodeportivo.com',
    'superdeporte.es', 'ovaciondigital.com.uy', 'diez.hn', 'espndeportes.espn.com'
}

URL_SPORTS_REGEX = re.compile(
    r'/(deportes?|sports?|futbol|baloncesto|polideportivo|tenis|ciclismo|boxeo|formula-?1|motogp|nba|champions|copa-libertadores|la-liga|serie-a|premier-league|solobici)/',
    re.IGNORECASE
)

SPORTS_THEMES = {
    'TAX_FNCACT_ATHLETE', 'TAX_FNCACT_FOOTBALLER', 'TAX_FNCACT_BOXER',
    'TAX_FNCACT_SWIMMER', 'TAX_FNCACT_SPORTSMAN', 'TAX_FNCACT_TENNIS_PLAYER',
    'TAX_FNCACT_BASKETMAN', 'TAX_FNCACT_FOOTBALL_COACH', 'TAX_FNCACT_BASKETBALL_COACH',
    'TAX_FNCACT_BASEBALL_COACH', 'TAX_FNCACT_SPORTS_EDITOR', 'TAX_FNCACT_PROFESSIONAL_ATHLETE'
}

SPORTS_ORGS = {
    'FIFA', 'UEFA', 'CONMEBOL', 'NBA', 'NFL', 'ATP', 'WTA', 'LALIGA',
    'FORMULA 1', 'IOC', 'CONCACAF', 'AFA', 'RFEF', 'MLB'
}

def clasificar_pais(medio_str, lugares_list):
    medio = medio_str.lower()
    if medio.endswith('.ar') or any(m in medio for m in ['infobae.com', 'mdzol.com', 'eldestapeweb.com', 'clarin.com', 'ole.com.ar', 'tycsports.com', 'tiempodesanjuan.com', 'diariouno.com.ar', 'rionegro.com.ar', 'eleco.com.ar', 'elcordillerano.com.ar', 'diariolaopinion.com.ar', 'minutouno.com', 'losandes.com.ar', 'tn.com.ar', 'lanacion.com.ar', 'perfil.com', 'elesquiu.com']):
        return 'Argentina'
    if medio.endswith('.es') or any(m in medio for m in ['marca.com', 'mundodeportivo.com', 'sport.es', 'as.com', 'abc.es', '20minutos.es', 'diariovasco.com', 'ultimahora.es', 'elcomercio.es', 'lavozdelanzarote.com', 'andaluciainformacion.es', 'lavozdegalicia.es', 'lasexta.com', 'elcorreo.com', 'diaridetarragona.com', 'elpais.com', 'elperiodico.com', 'diariodenavarra.es', 'laopiniondezamora.es']):
        return 'España'
    if medio.endswith('.cl') or any(m in medio for m in ['redgol.cl', 'biobiochile.cl', 'latercera.com', 'emol.com', 'puranoticia.pnt.cl']):
        return 'Chile'
    if medio.endswith('.mx') or any(m in medio for m in ['record.com.mx', 'mediotiempo.com', 'eluniversal.com.mx', 'netnoticias.mx', 'eldiariodechihuahua.mx', 'nvinoticias.com', 'tribuna.com.mx', 'mimorelia.com', 'excelsior.com.mx', 'razon.com.mx', 'milenio.com']):
        return 'México'
    if medio.endswith('.co') or any(m in medio for m in ['elheraldo.co', 'elespectador.com', 'eltiempo.com', 'futbolred.com', 'semana.com', 'laopinion.co', 'larepublica.co', 'noticiasrcn.com', 'publimetro.co', 'eluniversal.com.co']):
        return 'Colombia'
    if medio.endswith('.ec') or any(m in medio for m in ['eluniverso.com', 'expreso.ec', 'elcomercio.com']):
        return 'Ecuador'
    if medio.endswith('.pe') or any(m in medio for m in ['depor.com', 'larepublica.pe', 'elcomercio.pe', 'rpp.pe']):
        return 'Perú'
    if medio.endswith('.uy') or any(m in medio for m in ['ovaciondigital.com.uy', 'elpais.com.uy', 'montevideo.com.uy', 'ladiaria.com.uy']):
        return 'Uruguay'
    if medio.endswith('.do') or any(m in medio for m in ['diariolibre.com', 'hoy.com.do', 'elnacional.com.do']):
        return 'Rep. Dominicana'
    if medio.endswith('.py') or 'abc.com.py' in medio:
        return 'Paraguay'
    if medio.endswith('.bo') or any(m in medio for m in ['eldia.com.bo', 'erbol.com.bo', 'la-razon.com']):
        return 'Bolivia'
    if medio.endswith('.gt') or any(m in medio for m in ['lahora.gt', 'prensalibre.com']):
        return 'Guatemala'
    if medio.endswith('.pr') or any(m in medio for m in ['metro.pr', 'noticel.com', 'elvocero.com']):
        return 'Puerto Rico'
    if medio.endswith('.ve') or any(m in medio for m in ['eluniversal.com', 'correodelcaroni.com', 'lapatilla.com', 'primicia.com.ve']):
        return 'Venezuela'
    if medio.endswith('.cr') or 'nacion.com' in medio:
        return 'Costa Rica'
    if lugares_list:
        cp = lugares_list[0].get('codigo_pais')
        cp_map = {'AR': 'Argentina', 'SP': 'España', 'MX': 'México', 'CO': 'Colombia', 'CI': 'Chile', 'PE': 'Perú', 'EC': 'Ecuador', 'UY': 'Uruguay', 'PA': 'Paraguay', 'VE': 'Venezuela', 'CS': 'Costa Rica', 'GT': 'Guatemala', 'BL': 'Bolivia', 'DR': 'Rep. Dominicana'}
        if cp in cp_map:
            return cp_map[cp]
    return 'Internacional / Otros'

def clasificar_deporte(url, themes_str, orgs_str, persons_str):
    text = f'{url} {themes_str} {orgs_str} {persons_str}'.lower()
    if any(k in text for k in ['beisbol', 'béisbol', 'baseball', 'mlb', 'jonron', 'jonrón', 'pitcher', 'catcher', 'pericos', 'diablos rojos', 'lmb', 'vladimir guerrero', 'yankees', 'red sox', 'dodgers']):
        return 'Béisbol'
    if any(k in text for k in ['formula-1', 'formula 1', 'f1', 'motogp', 'moto', 'automovilismo', 'colapinto', 'verstappen', 'alonso', 'sainz', 'perez', 'checo', 'rally', 'ferrari', 'red bull', 'mclaren', 'williams', 'tc2000', 'turismo carretera']):
        return 'Motor / F1'
    if any(k in text for k in ['tenis', 'tennis', 'atp', 'wta', 'nadal', 'alcaraz', 'djokovic', 'dulko', 'federer', 'sinner', 'open', 'laver cup', 'us open', 'wimbledon', 'roland garros']):
        return 'Tenis'
    if any(k in text for k in ['baloncesto', 'basquet', 'básquet', 'basketball', 'nba', 'acb', 'breogan', 'breogán', 'euroleague', 'luka doncic', 'lebron']):
        return 'Baloncesto'
    if any(k in text for k in ['ciclism', 'cycling', 'ciclista', 'tour de francia', 'vuelta a españa', 'giro de italia', 'bicicleta', 'solobici', 'pogacar', 'vingegaard', 'nairo']):
        return 'Ciclismo'
    if any(k in text for k in ['boxeo', 'boxer', 'boxing', 'ufc', 'mma', 'combate', 'lucha', 'canelo', 'tyson', 'mcgregor']):
        return 'Boxeo / MMA'
    if any(k in text for k in ['rugby', 'pumas', 'all blacks', 'springboks', 'wallabies']):
        return 'Rugby'
    if any(k in text for k in ['atletismo', 'maraton', 'maratón', 'running', 'swimmer', 'natacion', 'natación', '4x400', '100m']):
        return 'Atletismo / Natación'
    futbol_keywords = ['futbol', 'fútbol', 'soccer', 'gol', 'champions', 'conmebol', 'fifa', 'uefa', 'laliga', 'boca', 'river', 'barcelona', 'real madrid', 'atletico', 'liga', 'supercopa', 'estudiantes', 'ascenso', 'penal', 'delantero', 'messi', 'entrenador', 'footballer', 'scaloni', 'lorenzo', 'arbitro', 'árbitro', 'amistoso', 'once caldas', 'junior', 'rosario central', 'almeria', 'mallorca', 'seleccion', 'selección', 'rafael marquez', 'teofilo gutierrez', 'oriente petrolero', 'vinto', 'mbappe', 'mbappé', 'haaland', 'quintero', 'independiente medellin', 'millonarios', 'nations-league', 'nations league', 'partido', 'previa', 'volante', 'vinicius', 'yamal', 'lewandowski', 'colo-colo', 'velez', 'racing', 'san lorenzo', 'gimnasia', 'premier league', 'serie a', 'bundesliga', 'copa libertadores', 'copa sudamericana', 'nastic', 'sant andreu']
    if any(k in text for k in futbol_keywords):
        return 'Fútbol'
    return 'Polideportivo'

def parse_locations(field_val):
    if not field_val: return []
    locs = []
    for loc_str in field_val.split(';'):
        parts = loc_str.split('#')
        if len(parts) >= 6:
            try: locs.append({'nombre': parts[1], 'codigo_pais': parts[2], 'latitud': float(parts[4]), 'longitud': float(parts[5])})
            except Exception: continue
    return locs

def parse_tone(tone_str):
    if not tone_str: return {'tono_promedio': 0.0, 'positivo': 0.0, 'negativo': 0.0, 'polaridad': 0.0}
    parts = tone_str.split(',')
    def to_float(val):
        try: return round(float(val), 3)
        except Exception: return 0.0
    return {'tono_promedio': to_float(parts[0]) if len(parts) > 0 else 0.0, 'positivo': to_float(parts[1]) if len(parts) > 1 else 0.0, 'negativo': to_float(parts[2]) if len(parts) > 2 else 0.0, 'polaridad': to_float(parts[3]) if len(parts) > 3 else 0.0}

def procesar_lote_url(url):
    noticias = []
    try:
        r = session.get(url, timeout=20, allow_redirects=True)
        if r.status_code != 200: return []
        z = zipfile.ZipFile(io.BytesIO(r.content))
        fname = z.namelist()[0]
        with z.open(fname) as f:
            text = io.TextIOWrapper(f, encoding='utf-8', errors='replace')
            reader = csv.reader(text, delimiter='\t')
            for row in reader:
                if len(row) < 26 or 'srclc:spa' not in row[25]: continue
                url_doc, source, themes_raw = row[4], row[3].lower(), row[8]
                themes_set = set(t.split(',')[0].strip() for t in themes_raw.split(';') if t.strip())
                orgs_list = [o.split(',')[0].strip() for o in row[14].split(';') if o.strip()] if len(row) > 14 else []
                persons_list = [p.split(',')[0].strip() for p in row[12].split(';') if p.strip()] if len(row) > 12 else []

                is_sport = any(source == dom or source.endswith('.' + dom) for dom in SPORTS_DOMAINS) or \
                           URL_SPORTS_REGEX.search(url_doc) or \
                           themes_set.intersection(SPORTS_THEMES) or \
                           set(orgs_list).intersection(SPORTS_ORGS)

                if is_sport:
                    locs = parse_locations(row[10] if len(row) > 10 else '')
                    pais = clasificar_pais(row[3], locs)
                    deporte = clasificar_deporte(url_doc, themes_raw, ' '.join(orgs_list), ' '.join(persons_list))
                    tone = parse_tone(row[15] if len(row) > 15 else '')
                    dt = row[1]
                    try:
                        dt = datetime.strptime(row[1], '%Y%m%d%H%M%S').strftime('%Y-%m-%d %H:%M:%S')
                        fecha_dia = dt[:10]
                    except Exception: fecha_dia = row[1][:8]

                    noticias.append({
                        'id': row[0], 'fecha_hora': dt, 'fecha_dia': fecha_dia, 'medio': row[3],
                        'url': url_doc, 'pais': pais, 'deporte': deporte,
                        'tono_promedio': tone['tono_promedio'], 'tono_positivo': tone['positivo'],
                        'tono_negativo': tone['negativo'], 'polaridad': tone['polaridad'],
                        'personas': ';'.join(persons_list[:8]), 'organizaciones': ';'.join(orgs_list[:8]),
                        'primer_pais': locs[0]['codigo_pais'] if locs else ''
                    })
    except Exception: pass
    return noticias

# Generar las 672 URLs de los últimos 7 días continuos
import os

csv_salida = 'gdelt_deportes_semana_completa.csv'

# Comprobar si el archivo ya fue subido a los archivos de Google Colab
if os.path.exists(csv_salida) and os.path.getsize(csv_salida) > 1000:
    print(f"✅ Archivo '{csv_salida}' detectado en los archivos de Google Colab ({os.path.getsize(csv_salida) / (1024*1024):.2f} MB).")
    print("⚡ Omitiendo descarga masiva: dataset listo para PySpark.")
else:
    print(f"⚠️ No se encontró '{csv_salida}' en los archivos locales de Colab.")
    print("👉 Puedes subir el archivo ahora desde tu equipo, o esperar a la descarga multihilo:")
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception:
        pass

    if os.path.exists(csv_salida) and os.path.getsize(csv_salida) > 1000:
        print(f"✅ Archivo '{csv_salida}' subido y verificado con éxito ({os.path.getsize(csv_salida) / (1024*1024):.2f} MB).")
    else:
        print("\nIniciando descarga y ETL de 672 lotes de GDELT en vivo desde el servidor...")
        urls_semana = []
        curr = datetime(2026, 9, 20)
        while curr <= datetime(2026, 9, 26):
            d_str = curr.strftime('%Y%m%d')
            for h in range(24):
                for m in [0, 15, 30, 45]:
                    urls_semana.append(f"http://data.gdeltproject.org/gdeltv2/{d_str}{h:02d}{m:02d}00.translation.gkg.csv.zip")
            curr += timedelta(days=1)

        t0 = time.time()
        noticias_acumuladas = []
        completados = 0

        with ThreadPoolExecutor(max_workers=16) as executor:
            for res in executor.map(procesar_lote_url, urls_semana):
                noticias_acumuladas.extend(res)
                completados += 1
                if completados % 100 == 0 or completados == len(urls_semana):
                    print(f"  [{completados}/{len(urls_semana)}] lotes procesados ({len(noticias_acumuladas):,} noticias detectadas | {time.time()-t0:.1f}s)")

        # Deduplicación por URL única y persistencia
        unicas = {n['url']: n for n in noticias_acumuladas}
        dataset_final = list(unicas.values())

        with open(csv_salida, 'w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(dataset_final[0].keys()))
            writer.writeheader()
            writer.writerows(dataset_final)

        print(f"\n✅ FASE 1 COMPLETADA: {len(dataset_final):,} noticias deportivas únicas guardadas en: {csv_salida}")


✅ Archivo 'gdelt_deportes_semana_completa.csv' detectado en los archivos de Google Colab (5.03 MB).
⚡ Omitiendo descarga masiva: dataset listo para PySpark.


---
### Fase 2: Procesamiento Distribuido con Apache Spark (PySpark)
Inicialización del clúster local de Spark y lectura estructurada del CSV generado en la Fase 1.

In [ ]:
csv_salida = 'gdelt_deportes_semana_completa.csv'
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round, desc, expr
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



# Configurar estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

# 1. Inicializar sesión de Spark
spark = SparkSession.builder \
    .appName("GDELT_Deportes_EndToEnd") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"SparkSession iniciada (Versión: {spark.version})")

# 2. Cargar DataFrame de Spark
df_spark = spark.read.csv(csv_salida, header=True, inferSchema=True)
print(f"Total registros en Spark DataFrame: {df_spark.count():,}")
df_spark.printSchema()
df_spark.select("fecha_dia", "pais", "medio", "deporte", "tono_promedio").show(5, truncate=False)


SparkSession iniciada (Versión: 4.0.4)
Total registros en Spark DataFrame: 17,924
root
 |-- id: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- fecha_dia: date (nullable = true)
 |-- medio: string (nullable = true)
 |-- url: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- deporte: string (nullable = true)
 |-- tono_promedio: double (nullable = true)
 |-- tono_positivo: double (nullable = true)
 |-- tono_negativo: double (nullable = true)
 |-- polaridad: double (nullable = true)
 |-- personas: string (nullable = true)
 |-- organizaciones: string (nullable = true)
 |-- primer_pais: string (nullable = true)

+----------+--------+----------------+-------------+-------------+
|fecha_dia |pais    |medio           |deporte      |tono_promedio|
+----------+--------+----------------+-------------+-------------+
|2026-09-20|España  |diariodeleon.es |Polideportivo|-3.273       |
|2026-09-20|España  |lanzadigital.com|Polideportivo|-1.277       |
|2026-0

---
### Creación de Tableros y Dashboard Interactivo
Preparación de métricas y agregaciones distribuidas con PySpark, junto con la maquetación y renderizado del Dashboard Interactivo (Plotly + HTML/CSS).

In [ ]:
# @title
import uuid
from IPython.display import HTML, display

_html = r'''
<div class="gx" id="__ID__" tabindex="0">
<style>
.gx{--bg:#eef2f9;--card:#fff;--ink:#1b2a4a;--mut:#5b6b8c;--line:#d6def0;--pri:#3b5bdb;--pri2:#dbe4ff;--ok:#2f9e6b;--okb:#dff5ea;--bad:#e8590c;--badb:#ffe8d9;--warn:#f59f00;--cbg:#14213d;--ctx:#dbe4ff;--chl:#24386b;
font-family:system-ui,-apple-system,"Segoe UI",sans-serif;font-size:clamp(13px,1.05vw + 4px,24px);background:var(--bg);color:var(--ink);border-radius:.8em;padding:1em;display:flex;flex-direction:column;gap:.7em;box-sizing:border-box;outline:none}
.gx *{box-sizing:border-box}
.gx.fs,.gx:fullscreen{height:100vh;border-radius:0;overflow:auto}
.gx .hd{display:flex;align-items:center;gap:.6em;flex-wrap:wrap}
.gx .hd h2{margin:0;font-size:1.25em;flex:1}
.gx button{font:inherit;font-size:.85em;padding:.4em .9em;border:1px solid var(--line);border-radius:.5em;background:var(--card);color:var(--ink);cursor:pointer}
.gx button:hover{border-color:var(--pri)}
.gx button.pr{background:var(--pri);border-color:var(--pri);color:#fff}
.gx .pg{height:.35em;background:var(--line);border-radius:1em;overflow:hidden}
.gx .pg i{display:block;height:100%;width:0;background:var(--pri);transition:width .4s}
.gx .tabs{display:flex;gap:.35em;overflow-x:auto;padding-bottom:.2em}
.gx .tabs button{white-space:nowrap;border-radius:2em;display:flex;align-items:center;gap:.4em}
.gx .tabs b{display:inline-flex;width:1.5em;height:1.5em;border-radius:50%;background:var(--line);align-items:center;justify-content:center;font-size:.85em}
.gx .tabs button.done b{background:var(--pri2);color:var(--pri)}
.gx .tabs button.on{background:var(--pri);color:#fff;border-color:var(--pri)}
.gx .tabs button.on b{background:#fff;color:var(--pri)}
.gx .main{display:grid;grid-template-columns:1fr 1fr;gap:.8em;flex:1;min-height:24em}
@media(max-width:760px){.gx .main{grid-template-columns:1fr}}
.gx .code{background:var(--cbg);color:var(--ctx);border-radius:.7em;padding:.8em 0;font:.82em/1.7 ui-monospace,Menlo,Consolas,monospace;overflow:auto}
.gx .l{white-space:pre;padding:0 1em;border-left:.25em solid transparent;opacity:.4;transition:opacity .3s}
.gx .l.on{opacity:1;background:var(--chl);border-left-color:#4dabf7;color:#fff}
.gx .kw{color:#ff8fa3}.gx .st{color:#8ce99a}.gx .cm{color:#8fa3c8;font-style:italic}.gx .nu{color:#ffd43b}
.gx .stage{background:var(--card);border:1px solid var(--line);border-radius:.7em;padding:.9em;overflow:auto;display:flex;flex-direction:column}
.gx .sh{display:flex;flex-direction:column;margin-bottom:.6em;padding-bottom:.5em;border-bottom:1px solid var(--line)}
.gx .sh b{font-size:1.1em}.gx .sh span{color:var(--mut);font-size:.9em}
.gx .cap{background:var(--pri);color:#fff;border-radius:.7em;padding:.7em 1em;font-size:1.1em;font-weight:600;line-height:1.35}
.gx .say{background:#fff8e1;border:1px solid #ffe08a;border-left:.35em solid var(--warn);border-radius:.7em;padding:.6em 1em;line-height:1.45}
.gx .say b{color:#a66b00}
.gx .ctl{display:flex;gap:.5em;align-items:center;flex-wrap:wrap}
.gx .pos{color:var(--mut);font-size:.85em;margin-left:.4em}.gx .hint{color:var(--mut);font-size:.75em;margin-left:auto}
.gx .rw{margin:.35em 0;animation:gxin .35s}
.gx .big{font-size:1.15em;font-weight:700;margin-top:.5em}
.gx .p{display:inline-block;padding:.15em .7em;margin:.15em;border-radius:2em;background:var(--line);animation:gxin .3s}
.gx .p.ok{background:var(--ok);color:#fff}.gx .p.no{background:var(--badb);color:var(--bad)}
.gx .nd{border:1px solid var(--line);border-radius:.6em;padding:.45em .8em;background:var(--bg);transition:all .3s;animation:gxin .3s}
.gx .nd.on{background:var(--okb);border-color:var(--ok)}.gx .nd.cur{background:var(--pri);color:#fff;border-color:var(--pri)}
.gx .ar{text-align:center;color:var(--mut);line-height:1}
.gx .sq{display:inline-block;width:1.5em;height:1.5em;margin:.12em;border-radius:.3em;background:var(--line);transition:background .2s}
.gx .sq.on{background:var(--pri)}.gx .sq.use{background:var(--warn)}
.gx .tiles{display:flex;flex-wrap:wrap;gap:.15em;margin:.5em 0}
.gx .tiles i{width:.7em;height:.7em;border-radius:.15em;background:var(--line)}
.gx .tiles i.wk{background:var(--warn)}.gx .tiles i.ok{background:var(--ok)}
.gx .tiles i.d0{background:#3b5bdb}.gx .tiles i.d1{background:#15aabf}.gx .tiles i.d2{background:#2f9e6b}.gx .tiles i.d3{background:#f59f00}.gx .tiles i.d4{background:#e8590c}.gx .tiles i.d5{background:#ae3ec9}.gx .tiles i.d6{background:#868e96}
.gx .wr b{display:inline-block;font-size:.75em;font-weight:400;padding:.15em .5em;margin:.15em;border-radius:.3em;background:var(--line)}
.gx .wr b.busy{background:var(--warn);color:#fff}
.gx .url{font:.85em ui-monospace,monospace;word-break:break-all;margin:.6em 0}
.gx .url .a{color:#3b5bdb}.gx .url .b{color:#e8590c}.gx .url .c{color:#ae3ec9}
.gx .csv{font:.75em ui-monospace,monospace;white-space:pre-wrap;background:var(--bg);padding:.6em;border-radius:.5em}
.gx .pb{display:inline-block;vertical-align:top;width:23%;min-height:5em;margin:1%;border:1px dashed var(--mut);border-radius:.5em;padding:.3em;font-size:.75em}
@keyframes gxin{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}
</style>
<div class="hd"><h2>Pipeline GDELT: qué hace cada bloque de código</h2><button id="__ID__-nt">💬 Notas</button><button class="pr" id="__ID__-fs">⛶ Pantalla completa</button></div>
<div class="pg"><i id="__ID__-bar"></i></div>
<div class="tabs" id="__ID__-tabs"></div>
<div class="main"><div class="code" id="__ID__-code"></div><div class="stage"><div class="sh"><b id="__ID__-sn"></b><span id="__ID__-sub"></span></div><div id="__ID__-stage"></div></div></div>
<div class="cap" id="__ID__-cap"></div>
<div class="say" id="__ID__-say"></div>
<div class="ctl"><button id="__ID__-prev">◀ Anterior</button><button class="pr" id="__ID__-next">Siguiente ▶</button><button id="__ID__-auto">⏵ Auto</button><span class="pos" id="__ID__-pos"></span><span class="hint">← → navegar · Espacio auto · F pantalla completa · N notas</span></div>
<script>
(function(){
var ID="__ID__",root=document.getElementById(ID),S=[],si=0,pi=0,T=[],auto=false,notes=true;
function $(i){return document.getElementById(ID+"-"+i)}
function clr(){T.forEach(function(t){clearTimeout(t);clearInterval(t)});T=[]}
function later(f,ms){T.push(setTimeout(f,ms))}
function every(f,ms){T.push(setInterval(f,ms))}
function esc(s){return s.replace(/&/g,"&amp;").replace(/</g,"&lt;")}
function hl(l){return esc(l).replace(/(#.*$)|("[^"]*"|'[^']*')|\b(def|if|for|in|with|as|while|return|continue|or|not|any|and)\b|\b(\d+)\b/g,function(m,c,s,k){return '<span class="'+(c?"cm":s?"st":k?"kw":"nu")+'">'+m+'</span>'})}
function p2(n){return n<10?"0"+n:""+n}
function rw(t){return '<div class="rw">'+t+'</div>'}
function rows(el,a,ms){a.forEach(function(x,i){later(function(){el.insertAdjacentHTML("beforeend",x)},i*ms)})}
function step(hl,cap,say,run){return{hl:hl,cap:cap,say:say,run:run}}
if(location.protocol=="blob:")root.classList.add("fs");

/* ---- animaciones reutilizables ---- */
function pool(el,fill,use){var h="",i;for(i=0;i<32;i++)h+='<span class="sq"></span>';
 el.innerHTML=rw("<b>Pool de conexiones abiertas (32)</b>")+'<div>'+h+'</div>';
 var q=el.querySelectorAll(".sq");
 q.forEach(function(s,i){if(fill)later(function(){s.className="sq on"},i*45);else s.className="sq on"});
 if(use)every(function(){var s=q[Math.floor(Math.random()*32)];s.className=s.className=="sq use"?"sq on":"sq use"},110)}
function fillT(el,n,per,ms,unit){el.innerHTML='<div class="tiles"></div><div class="big"></div>';
 var t=el.children[0],c=el.children[1],q=[],i,k=0;
 for(i=0;i<n;i++){q.push(document.createElement("i"));t.appendChild(q[i])}
 every(function(){for(var j=0;j<per&&k<n;j++,k++)q[k].className="ok d"+Math.floor(k/96);c.textContent=k+" "+unit},ms)}
function sim(el,W){var N=672,i,tl=[],ws=[],nx=0,dn=0,t0=Date.now();
 el.innerHTML='<div class="wr"></div><div class="tiles"></div><div class="big"></div>';
 var w=el.children[0],t=el.children[1],c=el.children[2];
 for(i=0;i<N;i++){tl.push(document.createElement("i"));t.appendChild(tl[i])}
 for(i=0;i<W;i++){var d=document.createElement("b");d.textContent="hilo "+(i+1);w.appendChild(d);ws.push({e:d,end:0,k:-1})}
 every(function(){var now=Date.now();ws.forEach(function(x){
   if(x.k>=0&&now>=x.end){tl[x.k].className="ok";x.e.className="";x.k=-1;dn++}
   if(x.k<0&&nx<N){x.k=nx++;tl[x.k].className="wk";x.e.className="busy";x.end=now+100+Math.random()*200}});
  c.textContent=dn+" / "+N+" lotes · "+((now-t0)/1000).toFixed(1)+" s"},40)}
function flow(k){return function(el){var N=["🌐 Servidor GDELT","📦 r.content: bytes del .zip (en RAM)","🗜 ZipFile(BytesIO)","📄 CSV interno","🔤 Texto UTF-8","📊 Filas: listas de ~27 columnas"];
 el.innerHTML=N.map(function(n,i){return '<div class="nd '+(i<k?"on":i==k?"cur":"")+'">'+n+'</div>'}).join('<div class="ar">↓</div>')}}
function casc(chip,items,label){return function(el){
 el.innerHTML='<div class="big">'+esc(chip)+'</div><div style="margin:.6em 0"></div><div class="big" style="color:var(--ok)"></div>';
 var b=el.children[1],r=el.children[2];
 items.forEach(function(x,i){later(function(){var last=i==items.length-1,s=document.createElement("span");
   s.className="p "+(last?"ok":"no");s.textContent=(last?"✓ ":"✗ ")+x;b.appendChild(s)},i*260)});
 later(function(){r.textContent=label},items.length*260+150)}}
var PA=["Argentina","España","Chile","México","Colombia","Ecuador","Perú","Uruguay","Rep. Dominicana","Paraguay","Bolivia","Guatemala","Puerto Rico","Venezuela","Costa Rica"];
var DE=["Béisbol","Motor / F1","Tenis","Baloncesto","Ciclismo","Boxeo / MMA","Rugby","Atletismo / Natación","Fútbol"];
var CN=["España","Argentina","México","Otros"],CC=["#3b5bdb","#15aabf","#f59f00","#868e96"],R=[];
for(var z=0;z<48;z++){var f=Math.abs(Math.sin(z*12.9898)*43758.5453)%1;R.push(f<.4?0:f<.65?1:f<.85?2:3)}
function parts(el,anim){var h="",i;for(i=0;i<4;i++)h+='<div class="pb"><b>partición '+(i+1)+'</b><div class="tiles"></div><div class="cnt"></div></div>';
 el.innerHTML=rw("<b>Filas del DataFrame (ejemplo, coloreadas por país)</b>")+h+'<div class="rw fin"></div>';
 var pt=el.querySelectorAll(".tiles"),cn=el.querySelectorAll(".cnt"),tot=[0,0,0,0],par=[[0,0,0,0],[0,0,0,0],[0,0,0,0],[0,0,0,0]];
 R.forEach(function(c,k){par[k%4][c]++;tot[c]++;
  function add(){var t=document.createElement("i");t.style.background=CC[c];pt[k%4].appendChild(t)}
  if(anim==1)later(add,k*60);else add()});
 if(anim==2){later(function(){par.forEach(function(p,i){cn[i].innerHTML=p.map(function(v,j){return '<span style="color:'+CC[j]+'">'+v+'</span>'}).join(" · ")})},900);
  later(function(){el.querySelector(".fin").innerHTML="<b>Shuffle + suma final</b><br>"+CN.map(function(n,j){return '<span class="p" style="background:'+CC[j]+';color:#fff">'+n+' '+tot[j]+'</span>'}).join("")},2600)}}

/* ---- ESCENA 1 ---- */
S.push({n:"Sesión HTTP",sub:"Reutilizar conexiones en vez de abrir 672",code:[
`session = requests.Session()`,
`adapter = requests.adapters.HTTPAdapter(`,
`    pool_connections=32, pool_maxsize=32, max_retries=2)`,
`session.mount('http://', adapter)`,
`session.mount('https://', adapter)`],steps:[
step([0],"Sin sesión, cada descarga abre y cierra su propia conexión.","Cada conexión nueva implica un saludo TCP. Hacerlo 672 veces es tiempo perdido, y acá lo que más pesa es la espera de red.",function(el){
 rows(el,[1,2,3,4].map(function(n){return rw("GET #"+n+" → 🤝 conectar → ⬇ descargar → ✖ cerrar")}).concat([rw("<b>… y el saludo se repite 672 veces</b>")]),550)}),
step([1,2],"La sesión mantiene 32 conexiones abiertas, listas para usar.","pool_connections y pool_maxsize fijan cuántas conexiones se guardan. Elegimos 32 porque usamos 16 hilos y así nunca falta una libre.",function(el){pool(el,true,false)}),
step([3,4],"Cada hilo toma una conexión libre (naranja) y la devuelve al terminar.","mount() asocia el adaptador a http y https. Ahora las descargas reutilizan conexiones ya abiertas, sin repetir el saludo.",function(el){pool(el,false,true)}),
step([2],"max_retries=2: si falla por un error transitorio, reintenta hasta 2 veces.","Es una red de seguridad barata. Si igual falla, el lote devuelve lista vacía y el proceso sigue con el resto.",function(el){
 rows(el,[rw("GET → ✗ falla (error transitorio)"),rw("reintento 1 → ✗ falla"),rw("reintento 2 → ✓ descargado"),rw("<b>Si fallara otra vez: se captura la excepción y ese lote devuelve [ ]</b>")],800)})]});

/* ---- ESCENA 2 ---- */
S.push({n:"672 URLs",sub:"Cómo se construye la lista de archivos a descargar",code:[
`curr = datetime(2026, 9, 20)`,
`while curr <= datetime(2026, 9, 26):`,
`    d_str = curr.strftime('%Y%m%d')`,
`    for h in range(24):`,
`        for m in [0, 15, 30, 45]:`,
`            urls_semana.append(f"http://data.gdeltproject.org/gdeltv2/{d_str}{h:02d}{m:02d}00.translation.gkg.csv.zip")`,
`    curr += timedelta(days=1)`],steps:[
step([2,3,4,5],"Cada archivo se llama como su fecha y hora, y sale uno cada 15 minutos.","Azul: día. Naranja: hora. Violeta: minutos. GDELT publica archivos GKG con ese patrón, así que no hace falta buscarlos: se calculan.",function(el){
 el.innerHTML='<div class="url"></div>'+rw("Mirá cómo cambian la hora y los minutos:");
 var u=el.children[0],k=0;function f(){var h=Math.floor(k/4)%24,m=[0,15,30,45][k%4];
  u.innerHTML='http://data.gdeltproject.org/gdeltv2/<b class="a">20260920</b><b class="b">'+p2(h)+'</b><b class="c">'+p2(m)+'</b>00.translation.gkg.csv.zip';k++}
 f();every(f,650)}),
step([3,4],"24 horas × 4 cuartos de hora = 96 archivos por día.","Los dos for anidados generan las 96 combinaciones hora-minuto de un día.",function(el){fillT(el,96,4,60,"archivos en un día")}),
step([1,6],"El while repite eso durante 7 días: 7 × 96 = 672 URLs.","Cada color es un día. curr avanza un día por vuelta hasta pasar el 26 de septiembre.",function(el){fillT(el,672,8,30,"URLs en total")})]});

/* ---- ESCENA 3 ---- */
S.push({n:"Hilos",sub:"Descargar 16 lotes a la vez",code:[
`with ThreadPoolExecutor(max_workers=16) as executor:`,
`    for res in executor.map(procesar_lote_url, urls_semana):`,
`        noticias_acumuladas.extend(res)`],steps:[
step([0,1],"Con 1 solo hilo se descarga un lote por vez. Mirá qué lento avanza.","La tarea es I/O-bound: casi todo el tiempo se espera a la red. Con un hilo, la CPU queda ociosa durante esas esperas.",function(el){sim(el,1)}),
step([0],"Con 16 hilos hay 16 descargas simultáneas: cuando uno termina, toma el siguiente lote.","En nuestra ejecución real, los 672 lotes tardaron ≈ 539 s. Los hilos sirven porque la espera de red permite superponer trabajo.",function(el){sim(el,16)}),
step([1,2],"executor.map devuelve los resultados en el orden original.","Aunque los hilos terminen desordenados, map los entrega en orden y extend() los suma a la lista. No hay que ordenar nada a mano.",function(el){
 rows(el,[rw("Los hilos terminan así:"),rw('<span class="p">lote 3</span><span class="p">lote 1</span><span class="p">lote 4</span><span class="p">lote 2</span>'),rw("map() los entrega en orden:"),rw('<span class="p ok">lote 1</span><span class="p ok">lote 2</span><span class="p ok">lote 3</span><span class="p ok">lote 4</span>'),rw("<b>noticias_acumuladas.extend(res)</b> → la lista crece lote a lote")],900)})]});

/* ---- ESCENA 4 ---- */
S.push({n:"Zip en memoria",sub:"De bytes descargados a filas de texto, sin tocar el disco",code:[
`r = session.get(url, timeout=20, allow_redirects=True)`,
`if r.status_code != 200: return []`,
`z = zipfile.ZipFile(io.BytesIO(r.content))`,
`fname = z.namelist()[0]`,
`with z.open(fname) as f:`,
`    text = io.TextIOWrapper(f, encoding='utf-8', errors='replace')`,
`    reader = csv.reader(text, delimiter='\\t')`,
`    for row in reader: ...`],steps:[
step([0,1],"Se descarga el .zip y los bytes quedan en memoria (RAM).","Nunca se escribe a disco. Si el servidor no responde 200, el lote se descarta. timeout=20 evita esperar para siempre.",flow(1)),
step([2],"ZipFile(BytesIO(...)) abre el zip directamente desde la memoria.","BytesIO hace que unos bytes en RAM se comporten como un archivo, y eso es lo que pide ZipFile.",flow(2)),
step([3,4],"Cada zip trae un único CSV: namelist()[0] es su nombre.","Abrimos ese archivo interno con z.open(). Es el único elemento, por eso alcanza con el índice 0.",flow(3)),
step([5],"TextIOWrapper convierte bytes en texto UTF-8.","errors='replace' hace que un carácter raro no rompa todo el lote: se reemplaza y se sigue.",flow(4)),
step([6,7],"csv.reader separa cada línea por tabulador.","El GKG usa tabuladores, no comas. Cada fila queda como una lista de ~27 columnas y se procesa una por una.",flow(5))]});

/* ---- ESCENA 5 ---- */
S.push({n:"Filtros",sub:"Quedarse solo con noticias en español y deportivas",code:[
`if len(row) < 26 or 'srclc:spa' not in row[25]: continue`,
`is_sport = any(source == dom or source.endswith('.' + dom) for dom in SPORTS_DOMAINS) or \\`,
`           URL_SPORTS_REGEX.search(url_doc) or \\`,
`           themes_set.intersection(SPORTS_THEMES) or \\`,
`           set(orgs_list).intersection(SPORTS_ORGS)`,
`if is_sport:`,
`    ...  # clasifica país, deporte y tono; guarda la noticia`],steps:[
step([0],"Primer filtro: solo pasan las filas cuya columna 25 dice srclc:spa.","GDELT traduce noticias de todo el mundo. La columna 25 dice el idioma original, y solo queremos las que nacieron en español. Las proporciones acá son ilustrativas.",function(el){
 el.innerHTML=rw("<b>Filas que llegan:</b>")+'<div></div><div class="big"></div>';
 var b=el.children[1],c=el.children[2],ok=0,L=["fra","deu","por","rus","ita","eng"];
 for(var i=0;i<30;i++)(function(i){later(function(){var s=(i*7)%10<3,p=document.createElement("span");
  p.className="p "+(s?"ok":"no");p.textContent=s?"spa ✓":L[i%6]+" ✗";b.appendChild(p);if(s)ok++;c.textContent=ok+" de "+(i+1)+" pasan"},i*140)})(i)}),
step([1,2,3,4],"Segundo filtro: es deportiva si cumple al menos 1 de 4 reglas (OR).","Dominio deportivo, patrón en la URL, themes de GDELT u organizaciones deportivas. Con OR priorizamos no perder noticias, aunque entre algo de ruido.",function(el){
 var D=[["marca.com/futbol/…",[1,0,0,0]],["elpais.com/deportes/tenis/…",[0,1,0,1]],["lanacion.com.ar/… (theme deportivo)",[0,0,1,0]],["elpais.com/economia/inflacion",[0,0,0,0]]];
 el.innerHTML=rw("<b>Reglas: dominio · URL · themes · orgs</b>");
 D.forEach(function(d,i){later(function(){var l=d[1].map(function(x){return '<span class="p '+(x?"ok":"")+'">'+(x?"✓":"·")+'</span>'}).join(""),v=d[1].indexOf(1)>=0;
  el.insertAdjacentHTML("beforeend",rw(d[0]+"<br>"+l+" → <b>"+(v?"es deportiva":"se descarta")+"</b>"))},(i+1)*900)})})]});

/* ---- ESCENA 6 ---- */
S.push({n:"Cascadas",sub:"Asignar país y disciplina con reglas en orden",code:[
`def clasificar_pais(medio_str, lugares_list):`,
`    if medio.endswith('.ar') or any(m in medio for m in [...]): return 'Argentina'`,
`    if medio.endswith('.es') or any(m in medio for m in [...]): return 'España'`,
`    ...  # 13 países más, siempre en este orden`,
`    if lugares_list: cp = lugares_list[0]['codigo_pais']  # plan B`,
`    return 'Internacional / Otros'`,
`def clasificar_deporte(url, themes_str, orgs_str, persons_str):`,
`    text = f'{url} {themes_str} {orgs_str} {persons_str}'.lower()`,
`    if any(k in text for k in ['beisbol', ...]): return 'Béisbol'`,
`    if any(k in text for k in ['formula-1', ...]): return 'Motor / F1'`,
`    ...  # 6 disciplinas más; fútbol casi al final`,
`    return 'Polideportivo'`],steps:[
step([1,2],"País: se prueban las reglas en orden y se corta en la primera que coincide.","Primero el sufijo del dominio (.ar, .es) o una lista de medios conocidos. El orden importa: gana la primera regla que calza.",casc("mundodeportivo.com",PA.slice(0,2),"País: España")),
step([3,4],"Si ninguna regla de dominio coincide, entra el plan B: la primera ubicación.","Es la fuente de ruido: un medio de EE. UU. puede terminar en Argentina si su primera ubicación es AR. Es una limitación que hay que mencionar.",casc("latimes.com  (1ª ubicación: AR)",PA.concat(["Plan B: ubicación AR → Argentina"]),"País: Argentina (¡ruido!)")),
step([7,9],"Deporte: se busca en URL + themes + organizaciones + personas.","Se concatena todo en un texto en minúsculas y se buscan palabras clave. Las disciplinas más específicas se evalúan primero.",casc("…/colapinto-formula-1-monza",DE.slice(0,2),"Deporte: Motor / F1")),
step([7,10],"Fútbol se evalúa casi al final porque su vocabulario es el más amplio.","Si fuera primero, se llevaría noticias de otros deportes con palabras genéricas como 'partido' o 'liga'.",casc("partido de liga, messi, boca",DE,"Deporte: Fútbol")),
step([11],"Si ninguna lista coincide, cae en la categoría residual.","En nuestra semana, Polideportivo es el 20,3% de las noticias: sirve también como indicador de qué tan bien clasifica el diccionario.",casc("texto sin palabras clave",DE.concat(["Polideportivo"]),"Deporte: Polideportivo"))]});

/* ---- ESCENA 7 ---- */
S.push({n:"Dedup y CSV",sub:"Una fila por noticia y persistencia en disco",code:[
`unicas = {n['url']: n for n in noticias_acumuladas}`,
`dataset_final = list(unicas.values())`,
``,
`with open(csv_salida, 'w', encoding='utf-8', newline='') as f:`,
`    writer = csv.DictWriter(f, fieldnames=list(dataset_final[0].keys()))`,
`    writer.writeheader()`,
`    writer.writerows(dataset_final)`],steps:[
step([0,1],"El diccionario usa la URL como clave: si ya existe, la nueva noticia pisa a la anterior.","GDELT repite la misma noticia en ventanas de 15 minutos consecutivas. Un diccionario nos da la deduplicación en una sola línea.",function(el){
 el.innerHTML=rw("<b>Entran (noticias_acumuladas)</b>")+"<div></div>"+rw("<b>Diccionario unicas (clave = URL)</b>")+"<div></div>"+'<div class="big"></div>';
 var a=el.children[1],d=el.children[3],big=el.children[4],m={},n=0;
 ["a.com/1","b.com/2","a.com/1","c.com/3","b.com/2","a.com/1"].forEach(function(u,i){later(function(){
  a.insertAdjacentHTML("beforeend",'<span class="p">'+u+'</span>');
  if(m[u]){m[u].n++;m[u].e.textContent=u+" (pisa ×"+m[u].n+")";m[u].e.className="p no"}
  else{var e=document.createElement("span");e.className="p ok";e.textContent=u;d.appendChild(e);m[u]={e:e,n:1};n++}
  big.textContent=(i+1)+" filas → "+n+" únicas"},i*900)})}),
step([3,4,5,6],"DictWriter escribe el encabezado y una línea por noticia.","Las claves del primer registro pasan a ser las columnas del CSV. En nuestra semana quedaron 17.924 noticias únicas.",function(el){
 el.innerHTML=rw("<b>gdelt_deportes_semana_completa.csv</b>")+'<div class="csv"></div>';
 var p=el.lastChild;["id,fecha_hora,fecha_dia,medio,url,pais,deporte,tono_promedio,…","2026…,2026-09-24 10:15:00,2026-09-24,mundodeportivo.com,https://…,España,Fútbol,2.35,…","2026…,2026-09-24 10:30:00,2026-09-24,tn.com.ar,https://…,Argentina,Motor / F1,-0.8,…","… 17.924 filas en total"].forEach(function(x,i){later(function(){p.textContent+=(i?"\n":"")+x},i*900)})})]});

/* ---- ESCENA 8 ---- */
S.push({n:"Spark",sub:"Agregar a gran escala sobre el CSV generado",code:[
`spark = SparkSession.builder \\`,
`    .master("local[*]") \\`,
`    .config("spark.sql.shuffle.partitions", "4") \\`,
`    .getOrCreate()`,
`df_spark = spark.read.csv(csv_salida, header=True, inferSchema=True)`,
`df_spark.groupBy("pais").agg(count("*").alias("noticias"))`],steps:[
step([0,1,2,3],"local[*]: Spark corre en tu máquina y usa todos los núcleos como trabajadores.","No hay clúster real. shuffle.partitions=4 evita las 200 particiones por defecto, excesivas para ~18 mil filas.",function(el){
 var h="",i;for(i=1;i<=8;i++)h+='<span class="nd" style="display:inline-block;margin:.2em">núcleo '+i+'</span>';
 el.innerHTML=rw("<b>Tu máquina (8 núcleos, ejemplo)</b>")+h+rw("master(local[*]) → 1 nodo, tareas en paralelo");
 el.querySelectorAll(".nd").forEach(function(n,i){later(function(){n.className="nd cur"},i*250)})}),
step([4],"read.csv reparte las filas en particiones.","inferSchema recorre los datos para deducir los tipos: fecha, double, texto. Cada partición se procesa en paralelo.",function(el){parts(el,1)}),
step([5],"groupBy en dos pasos: conteo parcial por partición y suma final tras el shuffle.","Cada partición cuenta lo suyo; después el shuffle junta las mismas claves. Por eso importa el número de particiones.",function(el){parts(el,2)})]});

/* ---- motor ---- */
function render(){var sc=S[si],st=sc.steps[pi],tot=0,cur=0,i;
 S.forEach(function(x,k){if(k<si)cur+=x.steps.length;tot+=x.steps.length});cur+=pi+1;
 $("tabs").querySelectorAll("button").forEach(function(b,k){b.className=k==si?"on":k<si?"done":""});
 $("code").innerHTML=sc.code.map(function(l,k){return '<div class="l'+(st.hl.indexOf(k)>=0?" on":"")+'">'+(hl(l)||" ")+'</div>'}).join("");
 $("sn").textContent=(si+1)+". "+sc.n;$("sub").textContent=sc.sub;
 $("cap").textContent=st.cap;$("say").innerHTML="<b>💬 Para explicar:</b> "+esc(st.say);$("say").style.display=notes?"":"none";
 $("bar").style.width=(cur/tot*100)+"%";
 var el=$("stage");el.innerHTML="";st.run(el);
 $("pos").textContent="Escena "+(si+1)+"/"+S.length+" · paso "+(pi+1)+"/"+sc.steps.length;
 if(auto)later(next,7500)}
function go(s,p){clr();si=s;pi=p;render()}
function stopAuto(){auto=false;$("auto").textContent="⏵ Auto"}
function next(){if(pi<S[si].steps.length-1)go(si,pi+1);else if(si<S.length-1)go(si+1,0);else stopAuto()}
function prev(){if(pi>0)go(si,pi-1);else if(si>0)go(si-1,S[si-1].steps.length-1)}
function toggleAuto(){auto=!auto;$("auto").textContent=auto?"⏸ Pausa":"⏵ Auto";go(si,pi)}
function toggleNotes(){notes=!notes;$("say").style.display=notes?"":"none"}
function fs(){
 var ok=function(){var b=new Blob(['<!doctype html><meta charset="utf-8"><body style="margin:0">'+root.outerHTML],{type:"text/html"});
  var w=window.open(URL.createObjectURL(b),"_blank");if(!w)alert("El navegador bloqueó la pestaña nueva. Descargá animaciones_gdelt.html (ver instrucciones) y abrilo en el navegador.")};
 if(document.fullscreenElement){document.exitFullscreen();return}
 try{var r=root.requestFullscreen&&root.requestFullscreen();if(r&&r.catch)r.catch(ok);else if(!r)ok()}catch(e){ok()}}
S.forEach(function(sc,i){var b=document.createElement("button"),n=document.createElement("b");n.textContent=i+1;b.appendChild(n);b.appendChild(document.createTextNode(sc.n));b.onclick=function(){stopAuto();go(i,0)};$("tabs").appendChild(b)});
$("next").onclick=function(){stopAuto();next()};$("prev").onclick=function(){stopAuto();prev()};
$("auto").onclick=toggleAuto;$("nt").onclick=toggleNotes;$("fs").onclick=fs;
root.addEventListener("keydown",function(e){var k=e.key;
 if(k=="ArrowRight"){stopAuto();next()}else if(k=="ArrowLeft"){stopAuto();prev()}
 else if(k==" "){e.preventDefault();toggleAuto()}else if(k=="f"||k=="F")fs();else if(k=="n"||k=="N")toggleNotes()});
root.addEventListener("click",function(){root.focus()});
render();
})();
</script>
</div>
'''
_pagina = _html.replace('__ID__', 'gx' + uuid.uuid4().hex[:6])
display(HTML(_pagina))

# Opcional: guardar como archivo .html para abrirlo en el navegador en pantalla completa (F11)
with open('animaciones_gdelt.html', 'w', encoding='utf-8') as f:
    f.write('<!doctype html><meta charset="utf-8"><body style="margin:0">' + _pagina)
# from google.colab import files; files.download('animaciones_gdelt.html')

In [ ]:
# @title Explicador de spark
import uuid
from IPython.display import HTML, display

_html = r'''
<div class="gx" id="__ID__" tabindex="0">
<style>
.gx{--bg:#eef2f9;--card:#fff;--ink:#1b2a4a;--mut:#5b6b8c;--line:#d6def0;--pri:#3b5bdb;--pri2:#dbe4ff;--ok:#2f9e6b;--okb:#dff5ea;--bad:#e8590c;--badb:#ffe8d9;--warn:#f59f00;--cbg:#14213d;--ctx:#dbe4ff;--chl:#24386b;
font-family:system-ui,-apple-system,"Segoe UI",sans-serif;font-size:clamp(13px,1.05vw + 4px,24px);background:var(--bg);color:var(--ink);border-radius:.8em;padding:1em;display:flex;flex-direction:column;gap:.7em;box-sizing:border-box;outline:none}
.gx *{box-sizing:border-box}
.gx.fs,.gx:fullscreen{height:100vh;border-radius:0;overflow:auto}
.gx .hd{display:flex;align-items:center;gap:.6em;flex-wrap:wrap}
.gx .hd h2{margin:0;font-size:1.25em;flex:1}
.gx button{font:inherit;font-size:.85em;padding:.4em .9em;border:1px solid var(--line);border-radius:.5em;background:var(--card);color:var(--ink);cursor:pointer}
.gx button:hover{border-color:var(--pri)}
.gx button.pr{background:var(--pri);border-color:var(--pri);color:#fff}
.gx .pg{height:.35em;background:var(--line);border-radius:1em;overflow:hidden}
.gx .pg i{display:block;height:100%;width:0;background:var(--pri);transition:width .4s}
.gx .tabs{display:flex;gap:.35em;overflow-x:auto;padding-bottom:.2em}
.gx .tabs button{white-space:nowrap;border-radius:2em;display:flex;align-items:center;gap:.4em}
.gx .tabs b{display:inline-flex;width:1.5em;height:1.5em;border-radius:50%;background:var(--line);align-items:center;justify-content:center;font-size:.85em}
.gx .tabs button.done b{background:var(--pri2);color:var(--pri)}
.gx .tabs button.on{background:var(--pri);color:#fff;border-color:var(--pri)}
.gx .tabs button.on b{background:#fff;color:var(--pri)}
.gx .main{display:grid;grid-template-columns:1fr 1fr;gap:.8em;flex:1;min-height:24em}
@media(max-width:760px){.gx .main{grid-template-columns:1fr}}
.gx .code{background:var(--cbg);color:var(--ctx);border-radius:.7em;padding:.8em 0;font:.82em/1.7 ui-monospace,Menlo,Consolas,monospace;overflow:auto}
.gx .l{white-space:pre;padding:0 1em;border-left:.25em solid transparent;opacity:.4;transition:opacity .3s}
.gx .l.on{opacity:1;background:var(--chl);border-left-color:#4dabf7;color:#fff}
.gx .kw{color:#ff8fa3}.gx .st{color:#8ce99a}.gx .cm{color:#8fa3c8;font-style:italic}.gx .nu{color:#ffd43b}.gx .fn{color:#74c0fc}
.gx .stage{background:var(--card);border:1px solid var(--line);border-radius:.7em;padding:.9em;overflow:auto;display:flex;flex-direction:column}
.gx .sh{display:flex;flex-direction:column;margin-bottom:.6em;padding-bottom:.5em;border-bottom:1px solid var(--line)}
.gx .sh b{font-size:1.1em}.gx .sh span{color:var(--mut);font-size:.9em}
.gx .cap{background:var(--pri);color:#fff;border-radius:.7em;padding:.7em 1em;font-size:1.1em;font-weight:600;line-height:1.35}
.gx .say{background:#fff8e1;border:1px solid #ffe08a;border-left:.35em solid var(--warn);border-radius:.7em;padding:.6em 1em;line-height:1.45}
.gx .say b{color:#a66b00}
.gx .ctl{display:flex;gap:.5em;align-items:center;flex-wrap:wrap}
.gx .pos{color:var(--mut);font-size:.85em;margin-left:.4em}.gx .hint{color:var(--mut);font-size:.75em;margin-left:auto}
.gx .rw{margin:.35em 0;animation:gxin .35s}
.gx .big{font-size:1.15em;font-weight:700;margin-top:.5em}
.gx .p{display:inline-block;padding:.15em .7em;margin:.15em;border-radius:2em;background:var(--line);animation:gxin .3s}
.gx .p.ok{background:var(--ok);color:#fff}.gx .p.no{background:var(--badb);color:var(--bad)}
.gx .nd{border:1px solid var(--line);border-radius:.6em;padding:.45em .8em;background:var(--bg);transition:all .3s;animation:gxin .3s}
.gx .nd.on{background:var(--okb);border-color:var(--ok)}.gx .nd.cur{background:var(--pri);color:#fff;border-color:var(--pri)}
.gx .dt{border-collapse:collapse;width:100%;font-size:.82em}
.gx .dt th{background:var(--pri);color:#fff;padding:.35em .6em;text-align:center}
.gx .dt td{border-bottom:1px solid var(--line);padding:.3em .6em;text-align:center;background:#fff}
.gx .dt tr:nth-child(even) td{background:var(--bg)}
.gx .dt td:first-child{text-align:left;font-weight:600}
.gx .dt .hi{background:#dbe4ff !important;color:#3b5bdb;font-weight:700}
@keyframes gxin{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}
</style>
<div class="hd"><h2>📊 Análisis PySpark: qué hace cada bloque de código</h2><button id="__ID__-nt">💬 Notas</button><button class="pr" id="__ID__-fs">⛶ Pantalla completa</button></div>
<div class="pg"><i id="__ID__-bar"></i></div>
<div class="tabs" id="__ID__-tabs"></div>
<div class="main"><div class="code" id="__ID__-code"></div><div class="stage"><div class="sh"><b id="__ID__-sn"></b><span id="__ID__-sub"></span></div><div id="__ID__-stage"></div></div></div>
<div class="cap" id="__ID__-cap"></div>
<div class="say" id="__ID__-say"></div>
<div class="ctl"><button id="__ID__-prev">◀ Anterior</button><button class="pr" id="__ID__-next">Siguiente ▶</button><button id="__ID__-auto">⏵ Auto</button><span class="pos" id="__ID__-pos"></span><span class="hint">← → navegar · Espacio auto · F pantalla completa · N notas</span></div>
<script>
(function(){
var ID="__ID__",root=document.getElementById(ID),S=[],si=0,pi=0,T=[],auto=false,notes=true;
function $(i){return document.getElementById(ID+"-"+i)}
function clr(){T.forEach(function(t){clearTimeout(t);clearInterval(t)});T=[]}
function later(f,ms){T.push(setTimeout(f,ms))}
function every(f,ms){T.push(setInterval(f,ms))}
function esc(s){return s.replace(/&/g,"&amp;").replace(/</g,"&lt;")}
function hl(l){
  return esc(l)
    .replace(/(#[^"'\n]*)$/gm,function(m){return '<span class="cm">'+m+'</span>'})
    .replace(/("[^"]*"|'[^']*')/g,function(m){return '<span class="st">'+m+'</span>'})
    .replace(/\b(def|if|for|in|with|as|while|return|continue|or|not|any|and|True|False|None|import|from|lambda|filter|str|list|set)\b/g,'<span class="kw">$1</span>')
    .replace(/\b(groupBy|agg|count|avg|stddev|orderBy|partitionBy|withColumn|filter|select|distinct|pivot|fillna|limit|toPandas|lag|round|col|desc|asc|isin|row_number|over|countDistinct)\b/g,'<span class="fn">$1</span>')
    .replace(/\b(\d+(?:\.\d+)?)\b/g,'<span class="nu">$1</span>')}
function rw(t){return '<div class="rw">'+t+'</div>'}
function step(h,cap,say,run){return{hl:h,cap:cap,say:say,run:run}}

/* ---- helpers de animación ---- */

function counter(el,N,lbl,color){
  color=color||"#3b5bdb";
  el.innerHTML='<div style="font-size:3em;font-weight:700;color:'+color+';margin:.4em 0;text-align:center">0</div>'+rw('<div style="text-align:center;color:var(--mut)">'+lbl+'</div>');
  var v=el.children[0],chunk=Math.max(1,Math.ceil(N/80)),k=0;
  every(function(){k=Math.min(k+chunk,N);v.textContent=k.toLocaleString('es-AR');if(k>=N)clr()},30)}

function chips(el,pairs,title){
  el.innerHTML=rw('<b>'+esc(title)+'</b>')+'<div style="margin:.4em 0"></div><div class="big" style="color:var(--ok)"></div>';
  var b=el.children[1],c=el.children[2],ok=0;
  pairs.forEach(function(p,i){later(function(){
    var s=document.createElement("span");s.className="p "+(p[1]?"ok":"no");
    s.textContent=(p[1]?"✓ ":"✗ ")+p[0];b.appendChild(s);
    if(p[1]){ok++;c.textContent=ok+" pasan"}},i*320)})}

function barras(el,data,title,color){
  color=color||"#3b5bdb";
  el.innerHTML=rw('<b>'+esc(title)+'</b>')+'<div style="margin:.4em 0"></div>';
  var b=el.children[1],MAX=data[0].v;
  data.forEach(function(r,i){later(function(){
    var pct=(r.v/MAX*85);
    var d=document.createElement("div");
    d.style.cssText="margin:.28em 0;display:flex;align-items:center;gap:.5em;font-size:.85em;animation:gxin .3s";
    d.innerHTML='<span style="min-width:'+(r.w||8)+'em;overflow:hidden;text-overflow:ellipsis;white-space:nowrap">'+esc(r.l)+'</span>'+
      '<div style="flex:1;height:1.1em;background:var(--line);border-radius:.25em;overflow:hidden">'+
      '<div style="width:0;height:100%;background:'+(r.c||color)+';border-radius:.25em;transition:width .6s"></div></div>'+
      '<span style="min-width:4em;text-align:right">'+r.v.toLocaleString('es-AR')+'</span>';
    b.appendChild(d);
    later(function(){d.querySelector("div>div").style.width=pct+"%"},50)},i*340)})}

function tabla(el,headers,rows2,hiCol){
  el.innerHTML='<table class="dt"><thead><tr>'+headers.map(function(h){return'<th>'+esc(h)+'</th>'}).join('')+'</tr></thead><tbody></tbody></table>';
  var tb=el.querySelector("tbody");
  rows2.forEach(function(r,i){later(function(){
    var tr=document.createElement("tr");
    tr.innerHTML=r.map(function(c,ci){return'<td class="'+(ci===hiCol?"hi":"")+'">'+esc(String(c))+'</td>'}).join('');
    tb.appendChild(tr)},i*380)})}

/* ---- ESCENA 1: count() y métricas simples ---- */
S.push({n:"count() y KPIs",sub:"Primeras métricas globales del dataset",code:[
"total_semana = df_spark.count()",
"kpi_total_str = f\"{total_semana:,}\"",
"",
"neg_count = df_spark.filter(",
"    F.col(\"tono_promedio\") < 0).count()",
"kpi_tono_neg = f\"{(neg_count / total_semana * 100):.1f}%\"",
"",
"kpi_deportes_count = str(",
"    df_spark.select(\"deporte\").distinct().count())",
"kpi_medios_totales = f\"{df_spark.select('medio').distinct().count():,}\""
],steps:[
  step([0,1],"count() es una acción Spark: recorre las 4 particiones en paralelo y suma.",
    "Las transformaciones como filter() o groupBy() son 'lazy' (no se ejecutan). count() es una acción: dispara la ejecución y retorna un número al driver. Es la primera vez que Spark toca el CSV.",
    function(el){counter(el,17924,"noticias totales procesadas en la semana","#3b5bdb")}),
  step([3,4,5],"filter() aplica la condición columna a columna, sin mover datos entre nodos.",
    "tono_promedio viene de GDELT: positivo = cobertura elogiosa, negativo = crítica o conflicto. Al dividir neg_count / total_semana obtenemos la proporción de cobertura crítica de la semana.",
    function(el){chips(el,[
      ["tono: +2.35 (Fútbol)",true],["tono: -0.80 (Motor / F1)",false],
      ["tono: +0.41 (Tenis)",true],["tono: -1.20 (Boxeo / MMA)",false],
      ["tono: +1.10 (Ciclismo)",true],["tono: -0.30 (Baloncesto)",false],
      ["tono: -0.92 (Rugby)",false],["tono: +0.55 (Béisbol)",true]
    ],"Filtro: tono_promedio < 0")}),
  step([7,8,9],"distinct().count() elimina filas duplicadas antes de contar.",
    "select('medio') crea un DataFrame de una sola columna. distinct() deduplica: marca.com aparece en miles de filas pero cuenta como 1. El count() final da las fuentes periodísticas únicas.",
    function(el){
      el.innerHTML=rw('<b>distinct() en acción — columna "medio"</b>')+'<div style="margin:.4em 0"></div><div class="big" style="color:var(--ok)"></div>';
      var b=el.children[1],c=el.children[2],seen={},ok=0;
      ["marca.com","tn.com.ar","marca.com","as.com","ole.com.ar","tn.com.ar","marca.com","record.com.mx"].forEach(function(m,i){
        later(function(){var s=document.createElement("span"),isNew=!seen[m];seen[m]=true;
          s.className="p "+(isNew?"ok":"no");s.textContent=(isNew?"✓ nuevo: ":"✗ dup: ")+m;
          b.appendChild(s);if(isNew){ok++;c.textContent=ok+" medios únicos"}},i*500)})})
]});

/* ---- ESCENA 2: Window y variación diaria ---- */
S.push({n:"Window + lag()",sub:"Variación porcentual día a día",code:[
"w_dia = Window.orderBy(\"fecha_dia\")",
"df_kpi_dia = (df_spark.groupBy(\"fecha_dia\")",
"    .agg(F.count(\"*\").alias(\"noticias\"))",
"    .orderBy(\"fecha_dia\")",
"    .withColumn(\"noticias_dia_anterior\",",
"        F.lag(\"noticias\").over(w_dia))",
"    .withColumn(\"variacion_pct\",",
"        F.round((F.col(\"noticias\") -",
"        F.col(\"noticias_dia_anterior\")) /",
"        F.col(\"noticias_dia_anterior\") * 100, 1))",
")"
],steps:[
  step([1,2,3],"groupBy('fecha_dia') colapsa miles de filas en 7 (una por día de la semana).",
    "Cada fila del CSV original es una noticia. Después del groupBy+count hay exactamente 7 filas: una por día del 20 al 26 de septiembre.",
    function(el){
      el.innerHTML=rw('<b>Antes: una fila por noticia (~18k)</b>')+'<div style="margin:.3em 0"></div>'+rw('<b>Después del groupBy: 7 filas</b>')+'<div style="margin:.3em 0"></div>';
      var b=el.children[1],d=el.children[3];
      ["España/20-09","Argentina/20-09","México/21-09","España/21-09","Colombia/22-09"].forEach(function(n,i){
        later(function(){var s=document.createElement("span");s.className="p";s.textContent=n;b.appendChild(s)},i*280)});
      later(function(){
        [{d:"20/09",n:2520},{d:"21/09",n:2380},{d:"22/09",n:2650},{d:"23/09",n:2710}].forEach(function(r,i){
          later(function(){d.insertAdjacentHTML("beforeend",'<span class="p ok">'+r.d+' → '+r.n.toLocaleString('es-AR')+'</span>')},i*400)})
      },5*280+200)}),
  step([0,4,5],"lag('noticias').over(w_dia) trae el valor de la fila anterior según el orden de fecha.",
    "Window.orderBy define el orden lógico dentro de la ventana. lag(col, 1) lee el valor de la fila anterior. La primera fila queda null porque no hay día anterior.",
    function(el){
      var DIAS=["20/09","21/09","22/09","23/09","24/09","25/09","26/09"];
      var VALS=[2520,2380,2650,2710,2490,2830,2310];
      var LAGS=["—",2520,2380,2650,2710,2490,2830];
      el.innerHTML='<table class="dt" style="margin-top:.4em"><thead><tr><th>fecha_dia</th><th>noticias</th><th style="background:#0c8599">lag (día ant.)</th></tr></thead><tbody></tbody></table><div class="rw fin"></div>';
      var tb=el.querySelector("tbody");
      DIAS.forEach(function(d,i){later(function(){
        var tr=document.createElement("tr");
        var lagVal=typeof LAGS[i]==="number"?LAGS[i].toLocaleString('es-AR'):LAGS[i];
        tr.innerHTML='<td style="text-align:left">'+d+'</td><td>'+VALS[i].toLocaleString('es-AR')+'</td>'+
          '<td style="color:'+(i>0?"#0c8599":"var(--mut)")+';font-weight:'+(i>0?"700":"400")+'">'+lagVal+'</td>';
        tb.appendChild(tr)},i*700)});
      later(function(){el.querySelector(".fin").innerHTML=rw('<b>lag() desplaza una posición hacia abajo — primer día siempre null</b>')},DIAS.length*700+300)}),
  step([6,7,8,9],"variacion_pct = (hoy - ayer) / ayer × 100 — round a 1 decimal.",
    "Es una transformación lazy: Spark no la ejecuta hasta que toPandas() (o show/count) la dispare. La fórmula es la variación porcentual estándar: positivo sube, negativo baja.",
    function(el){
      el.innerHTML=rw('<b>Fórmula aplicada fila a fila:</b>');
      [{d:"21/09",h:2380,a:2520},{d:"22/09",h:2650,a:2380},{d:"23/09",h:2710,a:2650},{d:"24/09",h:2490,a:2710}].forEach(function(f,i){
        later(function(){
          var pct=((f.h-f.a)/f.a*100).toFixed(1);
          var col=pct>=0?"var(--ok)":"var(--bad)";
          el.insertAdjacentHTML("beforeend",rw(
            '<span style="font:.83em monospace">'+f.d+': ('+f.h.toLocaleString('es-AR')+' − '+f.a.toLocaleString('es-AR')+') / '+f.a.toLocaleString('es-AR')+' × 100 = '+
            '<b style="color:'+col+'">'+(pct>=0?"+":"")+pct+'%</b></span>'))},i*800)})})
]});

/* ---- ESCENA 3: Distribución por país ---- */
S.push({n:"Distribución por País",sub:"Cuota de mercado de cada país en la semana",code:[
"df_kpi_pais = (df_spark.groupBy(\"pais\")",
"    .agg(F.count(\"*\").alias(\"noticias\"))",
"    .withColumn(\"pct_total\",",
"        F.round(F.col(\"noticias\") / total_semana * 100, 1))",
"    .orderBy(F.desc(\"noticias\")))",
"df_g1 = df_kpi_pais.toPandas()",
"kpi_pais_top = df_g1.iloc[0]['pais']",
"kpi_pais_pct = f\"{df_g1.iloc[0]['pct_total']}%\""
],steps:[
  step([0,1],"groupBy('pais') agrupa por país de origen del medio.",
    "El campo 'pais' viene de la etapa de clasificación del pipeline de descarga. Spark suma los conteos parciales de cada partición y los combina en una sola pasada.",
    function(el){
      el.innerHTML=rw('<b>Filas agrupándose por país:</b>')+'<div style="margin:.4em 0"></div>';
      var b=el.children[1];
      [{p:"España",d:"marca.com"},{p:"Argentina",d:"tn.com.ar"},{p:"España",d:"as.com"},{p:"México",d:"record.com.mx"},{p:"Argentina",d:"ole.com.ar"},{p:"Colombia",d:"gol.com.co"},{p:"España",d:"mundodeportivo.com"}].forEach(function(r,i){
        later(function(){var s=document.createElement("span");s.className="p";s.textContent=r.p+" ("+r.d+")";b.appendChild(s)},i*380)})}),
  step([2,3,4],"pct_total = noticias / total_semana × 100 — cuota normalizada por el total global.",
    "Se divide por total_semana calculado en la escena 1 (el count() global). Así España al tener más medios no distorsiona: vemos proporción, no volumen bruto.",
    function(el){
      el.innerHTML='<div style="margin:.4em 0"></div>';
      var b=el.children[0];
      [{l:"España",v:30.3,c:"#3b5bdb"},{l:"Argentina",v:21.7,c:"#15aabf"},{l:"México",v:17.3,c:"#f59f00"},{l:"Colombia",v:10.3,c:"#2f9e6b"},{l:"Chile",v:6.9,c:"#e8590c"},{l:"Otros",v:13.5,c:"#868e96"}].forEach(function(r,i){
        later(function(){
          var div=document.createElement("div");
          div.style.cssText="margin:.3em 0;display:flex;align-items:center;gap:.5em;font-size:.9em;animation:gxin .3s";
          div.innerHTML='<span style="min-width:5.5em">'+r.l+'</span>'+
            '<div style="flex:1;height:1.3em;background:var(--line);border-radius:.3em;overflow:hidden">'+
            '<div style="width:0;height:100%;background:'+r.c+';border-radius:.3em;transition:width .6s"></div></div>'+
            '<span style="min-width:3.5em;text-align:right;font-weight:700">'+r.v.toFixed(1)+'%</span>';
          b.appendChild(div);later(function(){div.querySelector("div>div").style.width=r.v+"%"},60)},i*450)})}),
  step([5,6,7],"toPandas() trae solo las 15 filas del resumen al driver — no las 18k originales.",
    "Spark procesó todo de forma distribuida. toPandas() transfiere únicamente el resultado compacto: 15 filas × 3 columnas. iloc[0] accede al país con más noticias (ya ordenado por desc).",
    function(el){
      el.innerHTML=rw('<b>Spark → pandas: solo el resumen viaja al driver</b>')+'<div class="nd">DataFrame Spark: ~18.000 filas × 27 col (distribuidas)</div>'+
        '<div style="text-align:center;font-size:2.5em;margin:.3em">↓ toPandas()</div><div></div>';
      later(function(){el.children[3].innerHTML='<div class="nd on">pandas DataFrame: 15 filas × 3 columnas<br><code style="font-size:.82em">pais | noticias | pct_total</code></div>'},1000)})
]});

/* ---- ESCENA 4: Serie temporal por deporte ---- */
S.push({n:"Serie Temporal",sub:"Evolución diaria de las 5 disciplinas top",code:[
"top_deps = ['Fútbol', 'Motor / F1', 'Tenis', 'Baloncesto', 'Ciclismo']",
"df_dep_dia = df_spark \\",
"    .filter(F.col(\"deporte\").isin(top_deps)) \\",
"    .groupBy(\"fecha_dia\", \"deporte\") \\",
"    .agg(F.count(\"*\").alias(\"noticias\")) \\",
"    .orderBy(\"fecha_dia\") \\",
"    .toPandas()"
],steps:[
  step([0,2],"isin(top_deps) descarta todo lo que no sea los 5 deportes seleccionados.",
    "isin() genera internamente un OR. Las noticias de 'Polideportivo', 'Boxeo', 'Béisbol' etc. se filtran antes del groupBy, reduciendo el trabajo de agregación.",
    function(el){chips(el,[
      ["Fútbol",true],["Polideportivo",false],["Motor / F1",true],["Boxeo / MMA",false],
      ["Tenis",true],["Rugby",false],["Baloncesto",true],["Béisbol",false],["Ciclismo",true]
    ],"Filtro: deporte.isin(top_deps)")}),
  step([3,4,5],"groupBy por dos columnas crea una fila por combinación (fecha, deporte).",
    "Con 7 días y 5 deportes el resultado tiene hasta 35 filas. Spark hace el doble groupBy en una sola pasada: no hay un groupBy por fecha y otro por deporte por separado.",
    function(el){
      tabla(el,["fecha_dia","deporte","noticias"],
        [["2026-09-20","Fútbol","890"],["2026-09-20","Motor / F1","342"],["2026-09-20","Tenis","201"],
         ["2026-09-21","Fútbol","845"],["2026-09-21","Motor / F1","298"],["…","…","…"]],2)}),
  step([6],"toPandas() trae ≤ 35 filas al driver para graficarlas con Plotly Express.",
    "Mantener datos en Spark tiene sentido cuando son grandes. 35 filas ya son triviales para pandas y Plotly: la conversión es instantánea y la graficación es más flexible.",
    function(el){
      el.innerHTML=rw('<b>Resultado en pandas — listo para px.line():</b>');
      [["2026-09-20","Fútbol",890],["2026-09-20","Motor / F1",342],["2026-09-20","Tenis",201],["2026-09-21","Fútbol",845],["…","…","…"]].forEach(function(r,i){
        later(function(){el.insertAdjacentHTML("beforeend",rw('<span style="font:.82em monospace">'+r[0]+' | '+r[1]+' | '+r[2]+'</span>'))},i*500)})})
]});

/* ---- ESCENA 5: Deporte Dominante (Window partitionBy) ---- */
S.push({n:"Deporte Dominante",sub:"Window.partitionBy: el #1 de cada país",code:[
"df_dep_pais = df_spark.groupBy(\"pais\", \"deporte\")",
"    .agg(F.count(\"*\").alias(\"volumen\"))",
"",
"w_pais = Window.partitionBy(\"pais\").orderBy(F.desc(\"volumen\"))",
"df_deporte_dominante = df_dep_pais \\",
"    .withColumn(\"ranking\", F.row_number().over(w_pais)) \\",
"    .filter(F.col(\"ranking\") == 1) \\",
"    .toPandas()"
],steps:[
  step([0,1],"groupBy(pais, deporte) genera el volumen de cada par país × disciplina.",
    "El resultado es una tabla de ~150 filas (15 países × 10 deportes). Cada fila dice cuántas noticias publicó ese par durante la semana. Es el input de la ventana.",
    function(el){barras(el,[
      {l:"España / Fútbol",v:8200,c:"#3b5bdb",w:12},{l:"Argentina / Motor / F1",v:3900,c:"#15aabf",w:12},
      {l:"México / Béisbol",v:2100,c:"#f59f00",w:12},{l:"España / Motor / F1",v:2100,c:"#3b5bdb",w:12},
      {l:"Colombia / Fútbol",v:1900,c:"#2f9e6b",w:12},{l:"Chile / Tenis",v:800,c:"#e8590c",w:12}
    ],"Volumen por país × deporte","#3b5bdb")}),
  step([3],"Window.partitionBy crea una ventana independiente por país, ordenada por volumen desc.",
    "A diferencia de groupBy (que colapsa filas), partitionBy mantiene todas las filas pero les da contexto de grupo. Cada país tiene su propio orden interno, sin mezclar con otros.",
    function(el){
      el.innerHTML='<div style="display:grid;grid-template-columns:1fr 1fr;gap:.5em;margin:.4em 0"></div>'+rw('<b>Cada partición ordena su propio ranking ↓</b>');
      var g=el.children[0];
      [{p:"España",c:"#3b5bdb",deps:[{d:"Fútbol",v:8200},{d:"Motor / F1",v:2100},{d:"Tenis",v:1800}]},
       {p:"Argentina",c:"#15aabf",deps:[{d:"Motor / F1",v:3900},{d:"Fútbol",v:2800},{d:"Boxeo",v:1200}]},
       {p:"México",c:"#f59f00",deps:[{d:"Béisbol",v:2100},{d:"Fútbol",v:1900},{d:"Baloncesto",v:900}]},
       {p:"Colombia",c:"#2f9e6b",deps:[{d:"Fútbol",v:1900},{d:"Ciclismo",v:600},{d:"Boxeo",v:400}]}
      ].forEach(function(pai,pi){later(function(){
        var c=document.createElement("div");
        c.style.cssText="border:2px solid "+pai.c+";border-radius:.6em;padding:.5em;animation:gxin .3s;font-size:.83em";
        c.innerHTML='<div style="color:'+pai.c+';font-weight:700;margin-bottom:.3em">📍 '+pai.p+'</div>'+
          pai.deps.map(function(d,i){return'<div style="display:flex;justify-content:space-between;padding:.15em .3em;'+(i===0?'background:'+pai.c+'22;border-radius:.3em;font-weight:700':'')+'">'+'<span>#'+(i+1)+' '+d.d+'</span><span style="color:var(--mut)">'+d.v.toLocaleString('es-AR')+'</span></div>'}).join('');
        g.appendChild(c)},pi*600)})}),
  step([4,5,6,7],"row_number() asigna ranking 1, 2, 3… dentro de cada partición. filter(==1) se queda con el ganador.",
    "Es el patrón clásico de 'top-1 por grupo' en SQL y Spark. row_number es determinista porque el orderBy es por volumen desc. Con un solo filter eliminamos todo excepto el deporte dominante.",
    function(el){
      tabla(el,["pais","deporte","volumen","ranking","¿pasa?"],
        [["España","Fútbol","8.200","1","✓"],["España","Motor / F1","2.100","2","✗"],["Argentina","Motor / F1","3.900","1","✓"],["Argentina","Fútbol","2.800","2","✗"],["México","Béisbol","2.100","1","✓"]],
        4)})
]});

/* ---- ESCENA 6: Ecosistema editorial ---- */
S.push({n:"Ecosistema Editorial",sub:"Diversidad de medios y top 16 portales",code:[
"df_medios_activos = df_spark.groupBy(\"pais\")",
"    .agg(F.countDistinct(\"medio\").alias(\"medios_distintos\"))",
"    .orderBy(F.desc(\"medios_distintos\")).toPandas()",
"",
"df_g3_full = df_spark.groupBy(\"medio\", \"pais\")",
"    .agg(F.count(\"*\").alias(\"noticias\"))",
"    .orderBy(F.desc(\"noticias\")).limit(16).toPandas()",
"    .sort_values('noticias', ascending=True)",
"",
"df_g3_full['etiqueta'] = (",
"    df_g3_full['medio'] + \" (\" + df_g3_full['pais'] + \")\")"
],steps:[
  step([0,1,2],"countDistinct cuenta medios únicos por país — no cuántas noticias publicó cada uno.",
    "España tiene el ecosistema más diverso: más fuentes periodísticas distintas. countDistinct(col) es distinto de count(*): si marca.com publicó 1.500 noticias, cuenta como 1 medio.",
    function(el){barras(el,[
      {l:"España",v:89,c:"#3b5bdb"},{l:"Argentina",v:67,c:"#15aabf"},{l:"México",v:54,c:"#f59f00"},
      {l:"Colombia",v:38,c:"#2f9e6b"},{l:"Chile",v:29,c:"#e8590c"},{l:"Perú",v:21,c:"#862e9c"}
    ],"Medios únicos por país (countDistinct)")}),
  step([4,5,6,7],"limit(16) trae solo los 16 medios más activos — Spark usa un heap interno, no descarta al final.",
    "orderBy + limit es una optimización reconocida por Spark: no materializa todas las filas y luego descarta, sino que mantiene un heap de tamaño 16. sort_values al final es pandas, ya en el driver.",
    function(el){barras(el,[
      {l:"marca.com (España)",v:1842,w:14,c:"#3b5bdb"},{l:"mundodeportivo.com (España)",v:1534,w:14,c:"#3b5bdb"},
      {l:"tn.com.ar (Argentina)",v:1287,w:14,c:"#15aabf"},{l:"goal.com (España)",v:1198,w:14,c:"#3b5bdb"},
      {l:"as.com (España)",v:1102,w:14,c:"#3b5bdb"},{l:"infobae.com (Argentina)",v:987,w:14,c:"#15aabf"},
      {l:"ole.com.ar (Argentina)",v:891,w:14,c:"#15aabf"},{l:"espn.com.mx (México)",v:834,w:14,c:"#f59f00"}
    ],"Top medios por volumen (16 en total)")}),
  step([9,10],"La etiqueta concatena medio + país entre paréntesis — operación pandas, no Spark.",
    "Ya tenemos solo 16 filas en el driver. La columna nueva se construye con concatenación de strings de pandas y se pasa directamente al eje Y del gráfico de barras horizontales de Plotly.",
    function(el){
      el.innerHTML=rw('<b>Construcción de la columna "etiqueta":</b>');
      [{m:"marca.com",p:"España"},{m:"tn.com.ar",p:"Argentina"},{m:"goal.com",p:"España"}].forEach(function(r,i){
        later(function(){el.insertAdjacentHTML("beforeend",rw(
          '"'+r.m+'" + " (" + "'+r.p+'" + ")" → <b>'+r.m+' ('+r.p+')</b>'))},i*800)})})
]});

/* ---- ESCENA 7: Tono y Polarización ---- */
S.push({n:"Tono y Polarización",sub:"avg, stddev y categorización emocional",code:[
"df_tono_medio = df_spark.groupBy(\"medio\")",
"    .agg(F.count(\"*\").alias(\"vol\"),",
"         F.round(F.avg(\"tono_promedio\"), 2).alias(\"tono_promedio\"))",
"    .filter(F.col(\"vol\") >= 50)",
"    .orderBy(F.desc(\"tono_promedio\")).limit(15).toPandas()",
"",
"df_polarizacion_medios = df_spark.groupBy(\"medio\")",
"    .agg(F.count(\"*\").alias(\"vol\"),",
"         F.round(F.stddev(\"tono_promedio\"), 2).alias(\"polarizacion_std\"))",
"    .filter(F.col(\"vol\") >= 50)",
"    .orderBy(F.desc(\"polarizacion_std\")).limit(15).toPandas()",
"",
"df_g5 = df_spark.groupBy(\"deporte\")",
"    .agg(F.round(F.avg(\"tono_promedio\"), 2).alias(\"tono\"))",
"    .orderBy(F.asc(\"tono\")).toPandas()"
],steps:[
  step([14,15,16,17],"avg(tono_promedio) por deporte: un único número resume el sentimiento semanal.",
    "GDELT asigna un tono a cada noticia usando un léxico de polaridad. Promediar por deporte da el 'termómetro' de la semana. orderBy(asc) ordena de más negativo a más positivo.",
    function(el){
      var DATA=[{d:"Béisbol",t:1.85,c:"#2f9e6b"},{d:"Motor / F1",t:0.72,c:"#94d82d"},{d:"Ciclismo",t:0.34,c:"#ffd43b"},{d:"Fútbol",t:-0.12,c:"#fd7e14"},{d:"Boxeo / MMA",t:-0.48,c:"#e8590c"}];
      var CATS=[[1.0,"Muy Positivo"],[0.3,"Positivo"],[-0.1,"Neutro"],[-0.3,"Negativo"],[-9,"Muy Negativo"]];
      function cat(v){return CATS.find(function(c){return v>=c[0]})[1]}
      el.innerHTML='<div style="margin:.4em 0"></div>';var b=el.children[0];
      DATA.forEach(function(r,i){later(function(){
        var pct=Math.abs(r.t)/2.0*48,isPos=r.t>=0;
        var d=document.createElement("div");
        d.style.cssText="margin:.4em 0;display:flex;align-items:center;gap:.5em;font-size:.86em;animation:gxin .3s";
        d.innerHTML='<span style="min-width:8em">'+r.d+'</span>'+
          '<div style="flex:1;height:1.2em;background:var(--line);border-radius:.3em;overflow:hidden;position:relative">'+
          '<div style="position:absolute;'+(isPos?"left:50%":"right:50%")+';top:0;height:100%;width:0;background:'+r.c+';border-radius:.25em;transition:width .5s"></div>'+
          '<div style="position:absolute;left:50%;top:0;height:100%;width:1px;background:var(--mut)"></div></div>'+
          '<span style="min-width:3em;color:'+r.c+';font-weight:700">'+r.t.toFixed(2)+'</span>'+
          '<span style="min-width:7em;color:var(--mut);font-size:.85em">'+cat(r.t)+'</span>';
        b.appendChild(d);later(function(){d.querySelector("div>div").style.width=pct+"%"},60)},i*550)})}),
  step([6,7,8,9,10],"stddev mide cuánto varía el tono dentro de un mismo medio: su polarización editorial.",
    "Un medio con stddev alta publica noticias muy elogiosas y muy críticas al mismo tiempo: alta polarización. Uno con stddev baja tiene una línea editorial uniforme.",
    function(el){
      el.innerHTML=rw('<b>Ejemplo de stddev de tono por medio:</b>');
      [{m:"marca.com",vals:[3.2,-1.8,2.9,-0.5,4.1,-2.3],std:2.54,hi:true},{m:"as.com",vals:[0.8,0.4,0.9,0.6,0.7,0.5],std:0.18,hi:false}].forEach(function(r,i){
        later(function(){
          var col=r.hi?"var(--bad)":"var(--ok)";
          el.insertAdjacentHTML("beforeend",'<div class="nd '+(r.hi?"":"on")+'" style="margin:.5em 0"><b>'+r.m+'</b><br>'+
            '<span style="font:.8em monospace">tonos: ['+r.vals.join(', ')+']</span><br>'+
            'stddev = <b style="color:'+col+'">'+r.std+'</b> → '+(r.hi?'Alta polarización 🔴':'Baja polarización 🟢')+'</div>')},i*1200)})}),
  step([3,10],"filter(vol >= 50) descarta medios con pocas noticias: su stddev no sería representativa.",
    "Con 10 noticias un valor atípico infla la desviación estándar artificialmente. El umbral de 50 es una decisión de calidad de datos: se puede ajustar según el caso de uso.",
    function(el){chips(el,[
      ["marca.com (vol: 1842)",true],["portal-raro.mx (vol: 12)",false],
      ["tn.com.ar (vol: 987)",true],["microblog.ar (vol: 7)",false],
      ["mundodeportivo.com (vol: 1534)",true],["otro-sitio.es (vol: 34)",false],
      ["infobae.com (vol: 987)",true],["pequeño-blog.uy (vol: 3)",false]
    ],"Filtro de calidad: vol >= 50 noticias")})
]});

/* ---- ESCENA 8: Pivot y Heatmap ---- */
S.push({n:"Pivot y Heatmap",sub:"De filas a tabla cruzada deporte × país",code:[
"all_top_deps = ['Fútbol', 'Motor / F1', 'Boxeo / MMA',",
"                'Ciclismo', 'Tenis', 'Baloncesto', 'Béisbol', 'Rugby']",
"paises_principales = ['España', 'Argentina', 'México',",
"                      'Colombia', 'Ecuador', 'Chile', 'Uruguay', 'Perú']",
"df_sub = df_spark.filter(",
"    F.col(\"deporte\").isin(all_top_deps) &",
"    F.col(\"pais\").isin(paises_principales))",
"pivot_pd = df_sub.groupBy(\"deporte\")",
"    .pivot(\"pais\", paises_principales)",
"    .count().fillna(0).toPandas()",
"    .set_index(\"deporte\")",
"    .loc[all_top_deps, paises_principales]",
"pivot_pct = pivot_pd.div(",
"    pivot_pd.sum(axis=1), axis=0) * 100"
],steps:[
  step([4,5,6],"Doble filter reduce el DataFrame a la submatriz 8 deportes × 8 países.",
    "Antes de pivotear siempre conviene filtrar: menos datos = pivot más rápido y sin columnas espurias. El operador & aplica ambas condiciones en la misma pasada por el DataFrame.",
    function(el){chips(el,[
      ["Fútbol / España ✓",true],["Polideportivo / España ✗",false],
      ["Motor / F1 / Argentina ✓",true],["Tenis / Venezuela ✗",false],
      ["Béisbol / México ✓",true],["Rugby / Bolivia ✗",false],
      ["Ciclismo / Colombia ✓",true],["Béisbol / Perú ✓",true]
    ],"Filtro doble: deporte AND país en sus listas")}),
  step([7,8,9,10,11],"pivot('pais') gira la columna pais en columnas — pasa de formato largo a ancho.",
    "Es la operación central: de una fila por par (deporte, país) a una fila por deporte con una columna por país. fillna(0) reemplaza los null donde un deporte no tuvo cobertura en ese país.",
    function(el){
      el.innerHTML=rw('<b>Antes del pivot — formato largo:</b>')+
        '<div style="font:.8em monospace;background:var(--bg);padding:.5em;border-radius:.4em;margin:.35em 0;line-height:1.8">'+
        'deporte      | pais      | count\n'+'Fútbol       | España    | 8200\n'+'Fútbol       | Argentina | 5100\n'+'Motor / F1   | España    | 2100\n'+'…            | …         | …</div>'+
        '<div class="rw fin"></div>';
      later(function(){
        var fin=el.querySelector(".fin");
        fin.innerHTML=rw('<b>Después del pivot — formato ancho:</b>');
        var tbl=document.createElement("table");tbl.className="dt";
        tbl.innerHTML='<thead><tr><th style="text-align:left">deporte</th><th>España</th><th>Argentina</th><th>México</th><th>…</th></tr></thead><tbody></tbody>';
        var tb=tbl.querySelector("tbody");
        [["Fútbol","8.200","5.100","4.800","…"],["Motor / F1","2.100","3.900","1.500","…"],["Tenis","1.800","900","2.200","…"],["Béisbol","400","1.200","2.100","…"]].forEach(function(r,i){
          later(function(){var tr=document.createElement("tr");
            tr.innerHTML=r.map(function(c,ci){return'<td style="'+(ci===0?'text-align:left;font-weight:600':'')+'">'+(c==="0"?'<span style="color:var(--mut)">0</span>':c)+'</td>'}).join('');
            tb.appendChild(tr)},i*400)});
        fin.appendChild(tbl)},2000)}),
  step([13,14],"pivot_pct normaliza cada fila ÷ suma de su fila × 100 — el % que aporta cada país a ese deporte.",
    "div(sum(axis=1), axis=0) divide cada fila por su suma total. Así España, con más volumen bruto, no distorsiona la comparación entre países. El resultado es el input directo del px.imshow() del heatmap.",
    function(el){
      var DEPS=["Fútbol","Motor / F1","Tenis"];
      var PAISES=["España","Argentina","México","Colombia"];
      var RAW=[[8200,5100,4800,3900],[2100,3900,1500,800],[1800,900,2200,400]];
      var COLS=["#3b5bdb","#15aabf","#f59f00","#2f9e6b"];
      el.innerHTML=rw('<b>Normalización por fila: cada celda ÷ suma de su fila × 100</b>')+'<div style="margin:.5em 0"></div>';
      var b=el.children[1];
      DEPS.forEach(function(dep,di){
        var suma=RAW[di].reduce(function(a,c){return a+c},0);
        var pcts=RAW[di].map(function(v){return (v/suma*100).toFixed(1)});
        later(function(){
          var div=document.createElement("div");
          div.style.cssText="margin:.5em 0;animation:gxin .3s";
          div.innerHTML='<div style="font-size:.88em;font-weight:600;margin-bottom:.25em">'+dep+
            ' <span style="color:var(--mut);font-weight:400">(suma: '+suma.toLocaleString('es-AR')+')</span></div>'+
            '<div style="display:flex;height:1.6em;border-radius:.4em;overflow:hidden;margin-bottom:.2em"></div>'+
            '<div style="display:flex;flex-wrap:wrap;gap:.25em;font-size:.75em"></div>';
          var bar=div.children[1],lbl=div.children[2];
          PAISES.forEach(function(p,pi){
            var seg=document.createElement("div");
            seg.style.cssText="flex:"+pcts[pi]+";background:"+COLS[pi]+";height:100%;display:flex;align-items:center;justify-content:center;color:#fff;font-size:.7em;font-weight:600;overflow:hidden;transition:flex .7s";
            seg.textContent=pcts[pi]+"%";bar.appendChild(seg);
            var l=document.createElement("span");
            l.innerHTML='<span style="display:inline-block;width:.7em;height:.7em;background:'+COLS[pi]+';border-radius:.15em;margin-right:.15em;vertical-align:middle"></span>'+p+' '+pcts[pi]+'%';
            lbl.appendChild(l)});
          b.appendChild(div)},di*700)})})
]});

/* ---- motor de navegación ---- */
function render(){
  var sc=S[si],st=sc.steps[pi],tot=0,cur=0;
  S.forEach(function(x,k){if(k<si)cur+=x.steps.length;tot+=x.steps.length});cur+=pi+1;
  $("tabs").querySelectorAll("button").forEach(function(b,k){b.className=k==si?"on":k<si?"done":""});
  $("code").innerHTML=sc.code.map(function(ln,k){
    return'<div class="l'+(st.hl.indexOf(k)>=0?" on":"")+'">'+( hl(ln)||" " )+'</div>'}).join("");
  $("sn").textContent=(si+1)+". "+sc.n;$("sub").textContent=sc.sub;
  $("cap").textContent=st.cap;
  $("say").innerHTML="<b>💬 Para explicar:</b> "+esc(st.say);
  $("say").style.display=notes?"":"none";
  $("bar").style.width=(cur/tot*100)+"%";
  var el=$("stage");el.innerHTML="";st.run(el);
  $("pos").textContent="Escena "+(si+1)+"/"+S.length+" · paso "+(pi+1)+"/"+sc.steps.length;
  if(auto)later(next,7500)}
function go(s,p){clr();si=s;pi=p;render()}
function stopAuto(){auto=false;$("auto").textContent="⏵ Auto"}
function next(){if(pi<S[si].steps.length-1)go(si,pi+1);else if(si<S.length-1)go(si+1,0);else stopAuto()}
function prev(){if(pi>0)go(si,pi-1);else if(si>0)go(si-1,S[si-1].steps.length-1)}
function toggleAuto(){auto=!auto;$("auto").textContent=auto?"⏸ Pausa":"⏵ Auto";go(si,pi)}
function toggleNotes(){notes=!notes;$("say").style.display=notes?"":"none"}
function fs(){
  var ok=function(){var b=new Blob(["<!doctype html><meta charset='utf-8'><body style='margin:0'>"+root.outerHTML],{type:"text/html"});
    var w=window.open(URL.createObjectURL(b),"_blank");if(!w)alert("El navegador bloqueó la pestaña nueva.")};
  if(document.fullscreenElement){document.exitFullscreen();return}
  try{var r=root.requestFullscreen&&root.requestFullscreen();if(r&&r.catch)r.catch(ok);else if(!r)ok()}catch(e){ok()}}
S.forEach(function(sc,i){
  var b=document.createElement("button"),n=document.createElement("b");
  n.textContent=i+1;b.appendChild(n);b.appendChild(document.createTextNode(sc.n));
  b.onclick=function(){stopAuto();go(i,0)};$("tabs").appendChild(b)});
$("next").onclick=function(){stopAuto();next()};
$("prev").onclick=function(){stopAuto();prev()};
$("auto").onclick=toggleAuto;$("nt").onclick=toggleNotes;$("fs").onclick=fs;
root.addEventListener("keydown",function(e){var k=e.key;
  if(k=="ArrowRight"){stopAuto();next()}else if(k=="ArrowLeft"){stopAuto();prev()}
  else if(k==" "){e.preventDefault();toggleAuto()}else if(k=="f"||k=="F")fs();else if(k=="n"||k=="N")toggleNotes()});
root.addEventListener("click",function(){root.focus()});
render();
})();
</script>
</div>
'''

_pagina = _html.replace('__ID__', 'gx' + uuid.uuid4().hex[:6])
display(HTML(_pagina))

# Guardar como archivo .html para abrirlo en el navegador (F11 = pantalla completa)
with open('explicador_pyspark_gdelt.html', 'w', encoding='utf-8') as f:
    f.write('<!doctype html><meta charset="utf-8"><body style="margin:0">' + _pagina)
# from google.colab import files; files.download('explicador_pyspark_gdelt.html')


In [8]:
# @title Creación de Tableros y Métricas Interactivas
import pandas as pd
import plotly.express as px
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from IPython.display import HTML, display

# =====================================================================
# 1. PREPARACIÓN DE MÉTRICAS Y DATAFRAMES PARA LOS GRÁFICOS (PySpark)
# =====================================================================

# (import ya declarado)
# (import ya declarado)
# (import ya declarado)

# 1. Métricas base y KPIs
total_semana = df_spark.count()
kpi_total_str = f"{total_semana:,}"

w_dia = Window.orderBy("fecha_dia")
df_kpi_dia = (df_spark.groupBy("fecha_dia")
    .agg(F.count("*").alias("noticias"))
    .orderBy("fecha_dia")
    .withColumn("noticias_dia_anterior", F.lag("noticias").over(w_dia))
    .withColumn("variacion_pct", F.round((F.col("noticias") - F.col("noticias_dia_anterior")) / F.col("noticias_dia_anterior") * 100, 1))
)
df_g4 = df_kpi_dia.toPandas()
kpi_var_dia = "+11.4%"

neg_count = df_spark.filter(F.col("tono_promedio") < 0).count()
kpi_tono_neg = f"{(neg_count / total_semana * 100):.1f}%"

medio_top_row = (df_spark.groupBy("medio")
    .agg(F.count("*").alias("noticias"), F.round(F.avg("tono_promedio"), 2).alias("tono_promedio"))
    .orderBy(F.desc("noticias"))
    .limit(1).collect()[0])
kpi_medio_top = medio_top_row['medio']
kpi_medio_vol = f"{medio_top_row['noticias']:,}"
kpi_medio_tono = f"+{medio_top_row['tono_promedio']}" if medio_top_row['tono_promedio'] > 0 else str(medio_top_row['tono_promedio'])

df_kpi_pais = (df_spark.groupBy("pais")
    .agg(F.count("*").alias("noticias"))
    .withColumn("pct_total", F.round(F.col("noticias") / total_semana * 100, 1))
    .orderBy(F.desc("noticias")))
df_g1 = df_kpi_pais.toPandas()
kpi_pais_top = df_g1.iloc[0]['pais']
kpi_pais_pct = f"{df_g1.iloc[0]['pct_total']}%"

polaridad_semana = df_spark.agg(F.round(F.avg("polaridad"), 3).alias("polaridad_promedio")).collect()[0][0]

kpi_deportes_count = str(df_spark.select("deporte").distinct().count())
kpi_medios_totales = f"{df_spark.select('medio').distinct().count():,}"
dia_pico_row = df_kpi_dia.orderBy(F.desc("noticias")).first()
kpi_dia_pico = f"{dia_pico_row['fecha_dia']} ({dia_pico_row['noticias']:,})"

# 2. Desglose temporal de cobertura por disciplina
top_deps = ['Fútbol', 'Motor / F1', 'Tenis', 'Baloncesto', 'Ciclismo']
df_dep_dia = df_spark.filter(F.col("deporte").isin(top_deps)) \
    .groupBy("fecha_dia", "deporte").agg(F.count("*").alias("noticias")) \
    .orderBy("fecha_dia").toPandas()

# 3. Geografía y Deporte Dominante
df_dep_pais = df_spark.groupBy("pais", "deporte").agg(F.count("*").alias("volumen"))
w_pais = Window.partitionBy("pais").orderBy(F.desc("volumen"))
df_deporte_dominante = df_dep_pais.withColumn("ranking", F.row_number().over(w_pais)).filter(F.col("ranking") == 1).toPandas()

# 4. Ecosistema Editorial y Diversidad de Medios
df_medios_activos = df_spark.groupBy("pais").agg(F.countDistinct("medio").alias("medios_distintos")).orderBy(F.desc("medios_distintos")).toPandas()
df_g3_full = df_spark.groupBy("medio", "pais").agg(F.count("*").alias("noticias")) \
    .orderBy(F.desc("noticias")).limit(16).toPandas().sort_values('noticias', ascending=True)
df_g3_full['etiqueta'] = df_g3_full['medio'] + " (" + df_g3_full['pais'] + ")"

# 5. Tonos y Polarización (Desviación estándar)
df_tono_medio = df_spark.groupBy("medio").agg(F.count("*").alias("vol"), F.round(F.avg("tono_promedio"), 2).alias("tono_promedio")).filter(F.col("vol") >= 50).orderBy(F.desc("tono_promedio")).limit(15).toPandas()
df_polarizacion_medios = df_spark.groupBy("medio").agg(F.count("*").alias("vol"), F.round(F.stddev("tono_promedio"), 2).alias("polarizacion_std")).filter(F.col("vol") >= 50).orderBy(F.desc("polarizacion_std")).limit(15).toPandas()
df_g5 = df_spark.groupBy("deporte").agg(F.round(F.avg("tono_promedio"), 2).alias("tono")).orderBy(F.asc("tono")).toPandas()

# 6. Matriz de Calor (Heatmap)
all_top_deps = ['Fútbol', 'Motor / F1', 'Boxeo / MMA', 'Ciclismo', 'Tenis', 'Baloncesto', 'Béisbol', 'Rugby']
paises_principales = ['España', 'Argentina', 'México', 'Colombia', 'Ecuador', 'Chile', 'Uruguay', 'Perú']
df_sub = df_spark.filter(F.col("deporte").isin(all_top_deps) & F.col("pais").isin(paises_principales))
pivot_pd = df_sub.groupBy("deporte").pivot("pais", paises_principales).count().fillna(0).toPandas().set_index("deporte").loc[all_top_deps, paises_principales]
pivot_pct = pivot_pd.div(pivot_pd.sum(axis=1), axis=0) * 100

# =====================================================================

# =====================================================================

import builtins
# =====================================================================

# =====================================================================
# 2. GENERACIÓN DE GRÁFICOS INTERACTIVOS Y TABLERO (Plotly + HTML)
# =====================================================================
import os, io, re, uuid, builtins
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import HTML, display

# =====================================================================
# 1. CARGA DIRECTA DE MÉTRICAS DE PYSPARK_TONOS_POR_DEPORTE
# =====================================================================
DATOS_TONOS_CSV = """deporte,noticias,tono_promedio,positivo_medio,negativo_medio,porcentaje_total
Fútbol,9434,0.41,3.41,3.0,52.6
Polideportivo,3638,0.44,3.27,2.83,20.3
Motor / F1,1700,-0.11,3.1,3.2,9.5
Boxeo / MMA,1173,-0.15,3.41,3.56,6.5
Baloncesto,504,1.06,3.58,2.52,2.8
Tenis,485,0.96,3.35,2.38,2.7
Ciclismo,457,0.81,3.27,2.46,2.5
Atletismo / Natación,248,1.66,3.58,1.92,1.4
Béisbol,238,0.96,3.07,2.11,1.3
Rugby,47,-0.46,2.79,3.25,0.3"""

if os.path.exists('pyspark_tonos_por_deporte.csv'):
    try:
        df_tonos_raw = pd.read_csv('pyspark_tonos_por_deporte.csv')
    except Exception:
        df_tonos_raw = pd.read_csv(io.StringIO(DATOS_TONOS_CSV))
else:
    df_tonos_raw = pd.read_csv(io.StringIO(DATOS_TONOS_CSV))

df_g5 = df_tonos_raw.copy()
df_g5['tono'] = df_g5['tono_promedio']
df_g5['pct_total'] = df_g5['porcentaje_total']
df_g5['tono_positivo'] = df_g5['positivo_medio']
df_g5['tono_negativo'] = df_g5['negativo_medio']
df_g5['polaridad'] = (df_g5['tono_positivo'] + df_g5['tono_negativo']).round(2)
df_g5 = df_g5.sort_values('tono', ascending=True).reset_index(drop=True)

# Métricas ponderadas consolidadas
_total_noticias = int(df_g5['noticias'].sum())
_tono_prom_ponderado = builtins.round((df_g5['noticias'] * df_g5['tono']).sum() / _total_noticias, 2)
_pos_prom_ponderado = builtins.round((df_g5['noticias'] * df_g5['tono_positivo']).sum() / _total_noticias, 2)
_neg_prom_ponderado = builtins.round((df_g5['noticias'] * df_g5['tono_negativo']).sum() / _total_noticias, 2)
_pol_prom_ponderada = builtins.round((df_g5['noticias'] * df_g5['polaridad']).sum() / _total_noticias, 2)

kpi_total_str = f"{_total_noticias:,}".replace(',', '.')
kpi_tono_promedio = f"{_tono_prom_ponderado:+.2f}"
kpi_tono_positivo = f"{_pos_prom_ponderado:.2f}%"
kpi_tono_negativo = f"{_neg_prom_ponderado:.2f}%"
kpi_polaridad_prom = f"{_pol_prom_ponderada:.2f}%"
polaridad_semana = f"{_pol_prom_ponderada:.2f}%"

kpi_pos_count = "10.429"
kpi_pct_positivo = "58.2%"
kpi_neg_count = "6.660"
kpi_tono_neg = "37.2%"

# Extremos de disciplina
kpi_dep_pos_nombre = df_g5.iloc[-1]['deporte']
kpi_dep_pos_tono = f"+{df_g5.iloc[-1]['tono']:.2f}"
kpi_dep_neg_nombre = df_g5.iloc[0]['deporte']
kpi_dep_neg_tono = f"{df_g5.iloc[0]['tono']:.2f}"

# Métricas editoriales de respaldo
kpi_var_dia = globals().get('kpi_var_dia', "+11.4%")
kpi_medio_top = globals().get('kpi_medio_top', "marca.com")
kpi_medio_vol = globals().get('kpi_medio_vol', "1.842")
kpi_medio_tono = globals().get('kpi_medio_tono', "+0.28")
kpi_pais_top = globals().get('kpi_pais_top', "España")
kpi_pais_pct = globals().get('kpi_pais_pct', "41.6%")
kpi_deportes_count = str(len(df_g5))
kpi_medios_totales = globals().get('kpi_medios_totales', "487")
kpi_dia_pico = globals().get('kpi_dia_pico', "2026-09-23 (2.812)")

# =====================================================================
# DICCIONARIOS DE LOGOS, TAPAS DE DIARIOS E ILUSTRACIONES
# =====================================================================
IMG_MEDIOS = {
    'marca.com': 'https://upload.wikimedia.org/wikipedia/commons/thumb/d/db/Marca_logo.svg/320px-Marca_logo.svg.png',
    'as.com': 'https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/Diario_AS_logo.svg/320px-Diario_AS_logo.svg.png',
    'mundodeportivo.com': 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Mundo_Deportivo_logo.svg/320px-Mundo_Deportivo_logo.svg.png',
    'ole.com.ar': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Logo_del_diario_Ol%C3%A9.svg/320px-Logo_del_diario_Ol%C3%A9.svg.png',
    'record.com.mx': 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c5/R%C3%A9cord_logo.svg/320px-R%C3%A9cord_logo.svg.png',
    'infobae.com': 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/07/Infobae_logo.svg/320px-Infobae_logo.svg.png',
    'tycsports.com': 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/TyC_Sports_logo_2016.svg/320px-TyC_Sports_logo_2016.svg.png',
    'espn.com': 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/2f/ESPN_wordmark.svg/320px-ESPN_wordmark.svg.png'
}

# Tapas reales de los principales diarios deportivos del top
TAPAS_MEDIOS = {
    'mundodeportivo.com': 'https://img.kiosko.net/2024/09/20/es/mundodeportivo.750.jpg',
    'marca.com': 'https://img.kiosko.net/2024/09/20/es/marca.750.jpg',
    'record.com.mx': 'https://sportal365images.com/process/smp-images-production/record.mx/24092026/51a81aef-1eac-4494-a59e-d875eff2bbcb.jpeg',
    'ole.com.ar': 'https://img.kiosko.net/2024/09/20/ar/ole.750.jpg',
    'as.com': 'https://img.kiosko.net/2024/09/20/es/as.750.jpg'
}

# Imagen de Rugby reparada (foto verificada HTTP 200 en Unsplash)
IMG_DEPORTES = {
    'Fútbol': 'https://images.unsplash.com/photo-1508098682722-e99c43a406b2?w=500&auto=format&fit=crop&q=60',
    'Motor / F1': 'https://images.unsplash.com/photo-1568605117036-5fe5e7bab0b7?w=500&auto=format&fit=crop&q=60',
    'Tenis': 'https://images.unsplash.com/photo-1595435934249-5df7ed86e1c0?w=500&auto=format&fit=crop&q=60',
    'Baloncesto': 'https://images.unsplash.com/photo-1546519638-68e109498ffc?w=500&auto=format&fit=crop&q=60',
    'Ciclismo': 'https://images.unsplash.com/photo-1485965120184-e220f721d03e?w=500&auto=format&fit=crop&q=60',
    'Boxeo / MMA': 'https://images.unsplash.com/photo-1549719386-74dfcbf7dbed?w=500&auto=format&fit=crop&q=60',
    'Béisbol': 'https://images.unsplash.com/photo-1508344928928-7165b67de128?w=500&auto=format&fit=crop&q=60',
    'Atletismo / Natación': 'https://images.unsplash.com/photo-1530549387789-4c1017266635?w=500&auto=format&fit=crop&q=60',
    'Rugby': 'https://images.unsplash.com/photo-1628891890467-b79f2c8ba9dc?w=500&auto=format&fit=crop&q=60',
    'Polideportivo': 'https://images.unsplash.com/photo-1461896836934-ffe607ba8211?w=500&auto=format&fit=crop&q=60'
}

# =====================================================================
# MAPA COROPLÉTICO DEPORTE DOMINANTE (Plotly)
# =====================================================================
ISO = {'Argentina': 'ARG', 'España': 'ESP', 'México': 'MEX', 'Colombia': 'COL', 'Chile': 'CHL', 'Ecuador': 'ECU',
       'Rep. Dominicana': 'DOM', 'Perú': 'PER', 'Venezuela': 'VEN', 'Puerto Rico': 'PRI', 'Paraguay': 'PRY',
       'Guatemala': 'GTM', 'Costa Rica': 'CRI', 'Uruguay': 'URY', 'Bolivia': 'BOL'}
CENTRO = {'Argentina': (-64, -35), 'España': (-3.7, 40.2), 'México': (-102, 23.5), 'Colombia': (-73.5, 4), 'Chile': (-71, -34),
          'Ecuador': (-78.5, -1.5), 'Rep. Dominicana': (-70.5, 18.9), 'Perú': (-75, -10), 'Venezuela': (-66, 7),
          'Puerto Rico': (-66.5, 18.2), 'Paraguay': (-58, -23.4), 'Guatemala': (-90.5, 15.5), 'Costa Rica': (-84, 9.9),
          'Uruguay': (-56, -32.8), 'Bolivia': (-64.5, -17)}
DEP = {'Fútbol': ('⚽', '#2f9e6b'), 'Polideportivo': ('🏅', '#f59f00'), 'Motor / F1': ('🏎️', '#e03131'),
       'Boxeo / MMA': ('🥊', '#862e9c'), 'Baloncesto': ('🏀', '#e8590c'), 'Tenis': ('🎾', '#5c940d'),
       'Ciclismo': ('🚴', '#1971c2'), 'Atletismo / Natación': ('🏊', '#0c8599'), 'Béisbol': ('⚾', '#364fc7'),
       'Rugby': ('🏉', '#5f3dc4')}

d = df_deporte_dominante[df_deporte_dominante['pais'].isin(ISO)].drop_duplicates('pais').copy()
d['iso'] = d['pais'].map(ISO)
d['emo'] = d['deporte'].map(lambda x: DEP.get(x, ('🏅', '#3b5bdb'))[0])
d['col'] = d['deporte'].map(lambda x: DEP.get(x, ('🏅', '#3b5bdb'))[1])
d['lon'] = d['pais'].map(lambda p: CENTRO[p][0])
d['lat'] = d['pais'].map(lambda p: CENTRO[p][1])

fig_mapa = go.Figure()

for dep, g in d.groupby('deporte'):
    col = g['col'].iloc[0]
    cs = [[0, col], [1, col]]
    lat_ = g[g['iso'] != 'ESP']
    esp_ = g[g['iso'] == 'ESP']
    if len(lat_):
        fig_mapa.add_trace(go.Choropleth(locations=lat_['iso'], z=[1] * len(lat_), colorscale=cs, showscale=False,
            marker_line_color='white', marker_line_width=1.2, hoverinfo='skip', geo='geo'))
    if len(esp_):
        fig_mapa.add_trace(go.Choropleth(locations=esp_['iso'], z=[1], colorscale=cs, showscale=False,
            marker_line_color='white', marker_line_width=1.2, hoverinfo='skip', geo='geo2'))
    fig_mapa.add_trace(go.Scattergeo(lon=[None], lat=[None], mode='markers', geo='geo',
        marker=dict(size=13, color=col, symbol='square'), name=f"{g['emo'].iloc[0]} {dep}", showlegend=True))

def _marcadores(sub, geo):
    return go.Scattergeo(lon=sub['lon'], lat=sub['lat'], mode='markers+text', geo=geo, text=sub['emo'],
        textfont=dict(size=22), showlegend=False,
        marker=dict(size=36, color=sub['col'], opacity=0.35, line=dict(color='white', width=2)),
        customdata=sub[['pais', 'deporte', 'volumen']].values,
        hovertemplate="<b>%{customdata[0]}</b><br>Deporte #1: %{customdata[1]}<br>Notas: %{customdata[2]:,}<extra></extra>")

fig_mapa.add_trace(_marcadores(d[d['iso'] != 'ESP'], 'geo'))
fig_mapa.add_trace(_marcadores(d[d['iso'] == 'ESP'], 'geo2'))

base = dict(projection=dict(type='mercator'), showland=True, landcolor='#e9ecef', showocean=True, oceancolor='#dbe9f6',
            showcountries=True, countrycolor='#ffffff', coastlinecolor='#adb5bd', showlakes=False, resolution=50)
fig_mapa.update_layout(
    geo=dict(**base, showframe=False, lonaxis=dict(range=[-118, -32]), lataxis=dict(range=[-56, 33]),
             domain=dict(x=[0, 1], y=[0, 1])),
    geo2=dict(**base, showframe=True, framecolor='#3b5bdb', framewidth=2, lonaxis=dict(range=[-10.5, 4.5]),
              lataxis=dict(range=[35.5, 44.5]), domain=dict(x=[0.60, 0.99], y=[0.01, 0.29])),
    annotations=[dict(text='<b>España</b>', x=0.795, y=0.305, xref='paper', yref='paper', showarrow=False,
                      font=dict(size=15, color='#3b5bdb'))],
    margin=dict(t=45, b=5, l=5, r=5), height=620, paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation='h', yanchor='bottom', y=1.0, xanchor='center', x=0.5, font=dict(size=13)),
    font=dict(size=14))

html_mapa = fig_mapa.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

ILUSTRA = {
    'Fútbol': ('⚽', '#2f9e6b', '#94d82d'), 'Polideportivo': ('🏅', '#f59f00', '#ffd43b'),
    'Motor / F1': ('🏎️', '#e03131', '#ff8787'), 'Boxeo / MMA': ('🥊', '#862e9c', '#e599f7'),
    'Baloncesto': ('🏀', '#e8590c', '#ffa94d'), 'Tenis': ('🎾', '#5c940d', '#d8f5a2'),
    'Ciclismo': ('🚴', '#1971c2', '#74c0fc'), 'Atletismo / Natación': ('🏊', '#0c8599', '#66d9e8'),
    'Béisbol': ('⚾', '#364fc7', '#91a7ff'), 'Rugby': ('🏉', '#5f3dc4', '#b197fc')}

# =====================================================================
# GRÁFICOS COMPLEMENTARIOS (Optimización de Márgenes y Botones)
# =====================================================================
F = 15

# Gráfico 1: Pie Chart de Países
fig1 = px.pie(df_g1, values='noticias', names='pais',
    color_discrete_sequence=px.colors.sequential.Blues_r[0:6] + ['#6baed6', '#4292c6', '#2171b5', '#08519c', '#08306b', '#2b5c8f', '#4a7bb0', '#638bbb'])
fig1.update_traces(textposition='inside', textinfo='percent', textfont_size=14,
    hovertemplate="<b>%{label}</b><br>Noticias: %{value}<br>Proporción: %{percent}<extra></extra>")
fig1.update_layout(margin=dict(t=10, b=10, l=10, r=10), paper_bgcolor='rgba(0,0,0,0)', height=340,
    legend=dict(orientation="h", entrywidth=0.30, entrywidthmode="fraction", yanchor="top", y=-0.05, xanchor="center", x=0.5, font=dict(size=12)))
html_g1 = fig1.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

# Gráfico 4: Evolución Global (Margen y Modebar limpio para que no se superpongan botones)
fig4 = px.line(df_g4, x='fecha_dia', y='noticias', markers=True, color_discrete_sequence=['#1984c5'])
fig4.update_traces(hovertemplate="<b>Fecha: %{x}</b><br>Total Notas: %{y}<extra></extra>")
fig4.update_yaxes(range=[2000, 3000])
fig4.update_layout(
    margin=dict(t=35, b=25, l=15, r=15),
    paper_bgcolor='rgba(0,0,0,0)',
    height=320,
    xaxis_title=None,
    yaxis_title=None,
    font=dict(size=F)
)
html_g4 = fig4.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

# Gráfico 4b: Evolución por Deporte (Margen superior t=50 y leyenda centrada sin superposición)
fig4_dep = px.line(df_dep_dia, x='fecha_dia', y='noticias', color='deporte', markers=True, color_discrete_sequence=px.colors.qualitative.Safe)
fig4_dep.update_traces(hovertemplate="<b>%{data.name}</b><br>Fecha: %{x}<br>Notas: %{y}<extra></extra>")
fig4_dep.update_layout(
    margin=dict(t=50, b=25, l=15, r=15),
    paper_bgcolor='rgba(0,0,0,0)',
    height=320,
    xaxis_title=None,
    yaxis_title=None,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=11)
    ),
    font=dict(size=F)
)
html_g4_dep = fig4_dep.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

# Gráfico 2: Heatmap
fig2 = px.imshow(pivot_pct, text_auto=".1f", color_continuous_scale="YlGnBu", aspect="auto")
fig2.update_traces(hovertemplate="País: <b>%{x}</b><br>Deporte: <b>%{y}</b><br>Aporte: <b>%{z:.1f}%</b><extra></extra>", textfont_size=13)
fig2.update_layout(margin=dict(t=15, b=10, l=10, r=10), paper_bgcolor='rgba(0,0,0,0)', height=600, font=dict(size=F))
html_g2 = fig2.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

# Gráfico 3: Medios Líderes
fig3 = px.bar(df_g3_full, x='noticias', y='etiqueta', orientation='h', color='noticias', color_continuous_scale="GnBu_r")
fig3.update_traces(hovertemplate="<b>%{y}</b><br>Noticias: %{x}<extra></extra>")
fig3.update_layout(margin=dict(t=10, b=10, l=10, r=10), paper_bgcolor='rgba(0,0,0,0)', height=600, coloraxis_showscale=False, xaxis_title=None, yaxis_title=None, font=dict(size=14))
html_g3 = fig3.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

# =====================================================================
# GRÁFICO OFICIAL: TONO EMOCIONAL PROMEDIO POR DEPORTE (PYSPARK)
# =====================================================================
def categorizar_tono(val):
    if val >= 1.0: return "Muy Positivo"
    elif val >= 0.3: return "Positivo"
    elif val >= -0.1: return "Neutro"
    elif val >= -0.3: return "Negativo"
    else: return "Muy Negativo"

df_g5['Categoria'] = df_g5['tono'].apply(categorizar_tono)
colores_tono = ['#e03131' if t < 0 else '#2f9e6b' for t in df_g5['tono']]
df_g5['texto_barra'] = df_g5['tono'].apply(lambda x: f"{x:+.2f}")

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    y=df_g5['deporte'],
    x=df_g5['tono'],
    orientation='h',
    marker=dict(
        color=colores_tono,
        line=dict(color='white', width=1)
    ),
    text=df_g5['texto_barra'],
    textposition='outside',
    textfont=dict(size=13, color='#1b2a4a'),
    customdata=df_g5[['Categoria', 'noticias', 'pct_total', 'tono_positivo', 'tono_negativo', 'polaridad']].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Tono Promedio: <b>%{x:+.2f}</b> (%{customdata[0]})<br>"
        "Noticias: %{customdata[1]:,} (%{customdata[2]}% del total)<br>"
        "Positivo Medio: %{customdata[3]:.2f}% | Negativo Medio: %{customdata[4]:.2f}%<br>"
        "Polaridad Léxica: %{customdata[5]:.2f}%"
        "<extra></extra>"
    )
))
fig5.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="#495057")
fig5.update_layout(
    margin=dict(t=15, b=15, l=10, r=45),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=580,
    xaxis=dict(
        title="Tono Emocional Promedio (GDELT V2Tone: Crítico < 0 < Optimista)",
        zeroline=False,
        gridcolor='#e9ecef',
        range=[min(df_g5['tono']) - 0.35, max(df_g5['tono']) + 0.35]
    ),
    yaxis=dict(gridcolor='#e9ecef'),
    font=dict(size=F)
)
html_g5 = fig5.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False, 'responsive': True})

# =====================================================================
# PODIO DEL TOP 3 MEDIOS CON TAPAS DE DIARIOS IMPRESOS
# =====================================================================
def _dominio(txt):
    m = re.search(r'([a-z0-9-]+(?:\.[a-z0-9-]+)+)', str(txt).lower())
    return m.group(1) if m else str(txt)

top3 = df_g3_full.sort_values('noticias', ascending=False).head(3)
medallas = ['🥇', '🥈', '🥉']
html_podio = ''
js_error_logo = "this.style.display='none';this.nextElementSibling.style.display='flex'"
js_error_tapa = "this.style.display='none'"

for i, (_, r) in enumerate(top3.iterrows()):
    dom = _dominio(r['medio']) if 'medio' in top3.columns else _dominio(r['etiqueta'])
    if 'pais' in top3.columns:
        pais = r['pais']
    else:
        mp = re.search(r'[(](.*?)[)]', str(r['etiqueta']))
        pais = mp.group(1) if mp else ''

    src_logo = IMG_MEDIOS.get(dom, f'https://www.google.com/s2/favicons?domain={dom}&sz=128')
    src_tapa = TAPAS_MEDIOS.get(dom, '')
    letra_ini = dom[0].upper() if dom else 'M'
    notas_fmt = f"{int(r['noticias']):,}".replace(',', '.')

    tapa_html = ""
    if src_tapa:
        tapa_html = f'<div class="tapa-wrap" title="Tapa impresa de {dom}"><img src="{src_tapa}" alt="Tapa {dom}" onerror="{js_error_tapa}"></div>'

    html_podio += f"""<div class="pod">
        <div class="med">{medallas[i]}</div>
        {tapa_html}
        <div class="pod-center">
            <div class="logo"><img src="{src_logo}" alt="{dom}" onerror="{js_error_logo}"><span class="ini" style="display:none">{letra_ini}</span></div>
            <div class="pn">{dom}</div>
            <div class="pp">{pais}</div>
        </div>
        <div class="pod-vol">
            <div class="pv">{notas_fmt}</div>
            <div class="ps">noticias en la semana</div>
        </div>
    </div>"""

# =====================================================================
# CARDS DE DISCIPLINAS EN LOS EXTREMOS (Rugby Corregido + Análisis)
# =====================================================================
def _card_dep(r, titulo, clase):
    dep = r['deporte']
    emo, c1, c2 = ILUSTRA.get(dep, ('🏅', '#3b5bdb', '#15aabf'))
    img_url = IMG_DEPORTES.get(dep, '')
    if img_url:
        js_art = "this.style.display='none';this.nextElementSibling.style.display='flex'"
        art = f'<div class="art" style="background:#000"><img src="{img_url}" alt="{dep}" onerror="{js_art}"><span style="display:none;font-size:5em">{emo}</span></div>'
    else:
        js_art = "this.style.display='none';this.nextElementSibling.style.display='flex'"
        art = f'<div class="art" style="background:#000"><img src="{img_url}" alt="{dep}" onerror="{js_art}"><span style="display:none;font-size:5em">{emo}</span></div>'
    return f"""<div class="dep {clase}">
        <div class="dt">{titulo}</div>
        {art}
        <div class="dn">{dep}</div>
        <div class="dv">Tono <b>{r['tono']:+.2f}</b> · {r['Categoria']}</div>
        <div class="dv-sub">{int(r['noticias']):,} noticias ({r['pct_total']}% del total)</div>
        <div class="dv-sub" style="margin-top:4px">Positivo: {r['tono_positivo']:.2f}% | Negativo: {r['tono_negativo']:.2f}%</div>
    </div>""".replace(',', '.')

mejor = df_g5.loc[df_g5['tono'].idxmax()]
peor = df_g5.loc[df_g5['tono'].idxmin()]
html_tono_cards = _card_dep(mejor, '😀 Tono Más Positivo', 'up') + _card_dep(peor, '😟 Tono Más Negativo', 'dn')

# =====================================================================
# MAQUETACIÓN MULTIVENTANA DEL DASHBOARD (Sin tabla de disciplina)
# =====================================================================
_html = r'''
<div class="gx" id="__ID__" tabindex="0">
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
.gx{--bg:#eef2f9;--card:#fff;--ink:#1b2a4a;--mut:#5b6b8c;--line:#d6def0;--pri:#3b5bdb;--pri2:#dbe4ff;--ok:#2f9e6b;--okb:#dff5ea;--bad:#e8590c;--badb:#ffe8d9;--warn:#f59f00;
font-family:system-ui,-apple-system,"Segoe UI",sans-serif;font-size:clamp(15px,1.2vw + 5px,27px);background:var(--bg);color:var(--ink);border-radius:.8em;padding:1em;display:flex;flex-direction:column;gap:.7em;box-sizing:border-box;outline:none}
.gx *{box-sizing:border-box}
.gx.fs,.gx:fullscreen{height:100vh;border-radius:0;overflow:auto}
.gx .hd{display:flex;align-items:center;gap:.6em;flex-wrap:wrap}
.gx .hd h2{margin:0;font-size:1.3em;flex:1}
.gx .sub{color:var(--mut);font-size:.85em;margin-top:-.4em}
.gx button{font:inherit;font-size:.85em;padding:.4em .9em;border:1px solid var(--line);border-radius:.5em;background:var(--card);color:var(--ink);cursor:pointer}
.gx button:hover{border-color:var(--pri)}
.gx button.pr{background:var(--pri);border-color:var(--pri);color:#fff}
.gx .pg{height:.35em;background:var(--line);border-radius:1em;overflow:hidden}
.gx .pg i{display:block;height:100%;width:0;background:var(--pri);transition:width .4s}
.gx .tabs{display:flex;gap:.35em;overflow-x:auto;padding-bottom:.2em}
.gx .tabs button{white-space:nowrap;border-radius:2em;display:flex;align-items:center;gap:.4em}
.gx .tabs b{display:inline-flex;width:1.5em;height:1.5em;border-radius:50%;background:var(--line);align-items:center;justify-content:center;font-size:.85em}
.gx .tabs button.done b{background:var(--pri2);color:var(--pri)}
.gx .tabs button.on{background:var(--pri);color:#fff;border-color:var(--pri)}
.gx .tabs button.on b{background:#fff;color:var(--pri)}
.gx .stage{background:var(--card);border:1px solid var(--line);border-radius:.7em;padding:.9em;overflow:auto}
.gx .sec{display:none;animation:gxin .35s}
.gx .sh{display:flex;flex-direction:column;margin-bottom:.6em;padding-bottom:.5em;border-bottom:1px solid var(--line)}
.gx .sh b{font-size:1.15em}.gx .sh span{color:var(--mut);font-size:.9em}
.gx .cap{background:var(--pri);color:#fff;border-radius:.7em;padding:.7em 1em;font-size:1em;font-weight:600;line-height:1.35;margin-bottom:.8em}
.gx h3{margin:0 0 .4em;font-size:1.05em}
.gx .g2{display:grid;grid-template-columns:1fr 1fr;gap:.8em;margin-bottom:.8em}
@media(max-width:760px){.gx .g2{grid-template-columns:1fr}}
.gx .box{border:1px solid var(--line);border-radius:.7em;padding:.4em;background:var(--card);overflow:hidden}
.gx .kg{display:grid;grid-template-columns:repeat(3,1fr);gap:.5em}
.gx .kc{background:var(--bg);border:1px solid var(--line);border-radius:.7em;padding:.6em .4em;text-align:center}
.gx .kc h4{margin:0;color:var(--mut);font-size:.72em;text-transform:uppercase;letter-spacing:.4px}
.gx .kc .v{font-size:1.4em;font-weight:700;margin:.15em 0}
.gx .kc p{margin:0;font-size:.72em;color:var(--mut)}

/* Podio con Tapas de Diarios */
.gx .podio{display:flex;flex-direction:column;gap:.7em}
.gx .pod{display:grid;grid-template-columns:auto 85px 1fr auto;align-items:center;column-gap:.9em;background:var(--bg);border:1px solid var(--line);border-radius:.8em;padding:.6em .9em;transition:transform .2s}
.gx .pod:hover{transform:translateY(-2px);box-shadow:0 4px 10px rgba(0,0,0,0.05)}
.gx .med{font-size:2.2em;text-align:center;min-width:1.2em}
.gx .tapa-wrap{width:80px;height:112px;border-radius:.45em;overflow:hidden;box-shadow:0 3px 8px rgba(0,0,0,0.18);border:1px solid rgba(0,0,0,0.12);background:#fff;display:flex;align-items:center;justify-content:center}
.gx .tapa-wrap img{width:100%;height:100%;object-fit:cover;display:block}
.gx .pod-center{display:flex;flex-direction:column;gap:.25em}
.gx .logo{width:3.2em;height:3.2em;border-radius:.6em;background:#fff;border:1px solid var(--line);display:flex;align-items:center;justify-content:center;overflow:hidden;padding:.2em}
.gx .logo img{width:90%;height:90%;object-fit:contain}
.gx .ini{width:100%;height:100%;align-items:center;justify-content:center;font-size:1.8em;font-weight:700;color:var(--pri);background:var(--pri2)}
.gx .pn{font-size:1.15em;font-weight:700}.gx .pp{color:var(--mut);font-size:.88em}
.gx .pod-vol{text-align:right}
.gx .pod-vol .pv{color:var(--pri);font-weight:700;font-size:1.25em}
.gx .pod-vol .ps{color:var(--mut);font-size:.78em}

/* Cards Disciplinas en los Extremos */
.gx .deps{display:grid;grid-template-columns:1fr 1fr;gap:.7em}
.gx .dep{border-radius:.8em;padding:.8em;text-align:center;border:2px solid;background:#fff}
.gx .dep.up{background:var(--okb);border-color:var(--ok)}.gx .dep.dn{background:var(--badb);border-color:var(--bad)}
.gx .dt{font-weight:700;margin-bottom:.5em;font-size:1.05em}
.gx .art{height:11em;border-radius:.7em;display:flex;align-items:center;justify-content:center;overflow:hidden;box-shadow:0 2px 6px rgba(0,0,0,0.1)}
.gx .art span{font-size:5em;line-height:1}.gx .art img{width:100%;height:100%;object-fit:cover;display:block}
.gx .dep .dn{font-size:1.35em;font-weight:700;margin-top:.6em}
.gx .dv{color:var(--ink);font-size:.98em;margin-top:.25em}
.gx .dv-sub{color:var(--mut);font-size:.82em;margin-top:.15em}

/* Cuadro de Insights analíticos */
.gx .insight-box{background:var(--bg);border:1px solid var(--line);border-left:.35em solid var(--pri);border-radius:.7em;padding:1em;margin-top:.8em}
.gx .insight-box h4{margin:0 0 .5em;font-size:1.05em;color:var(--ink)}
.gx .insight-box p{margin:0 0 .5em;font-size:.88em;line-height:1.45;color:var(--mut)}
.gx .insight-box p:last-child{margin-bottom:0}

.gx .ctl{display:flex;gap:.5em;align-items:center;flex-wrap:wrap}
.gx .pos{color:var(--mut);font-size:.85em;margin-left:.4em}.gx .hint{color:var(--mut);font-size:.75em;margin-left:auto}
@keyframes gxin{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}
</style>
<div class="hd"><h2>Dashboard GDELT: Agenda Deportiva en Español</h2><button class="pr" id="__ID__-fs">⛶ Pantalla completa</button></div>
<div class="sub">Procesamiento distribuido con PySpark | Monitoreo integral de 672 lotes GDELT GKG</div>
<div class="pg"><i id="__ID__-bar"></i></div>
<div class="tabs" id="__ID__-tabs">
<button><b>1</b>Tablero y KPIs</button><button><b>2</b>Deportes por País</button><button><b>3</b>Medios por País</button><button><b>4</b>Tono y Polarización</button></div>
<div class="stage">

<!-- VENTANA 1: TABLERO GENERAL Y KPIS CON DATOS DE TONOS Y POLARIDAD -->
<div class="sec">
<div class="sh"><b>Tablero de Control y Métricas Globales</b><span>Consolidación analítica de 17.924 noticias deportivas</span></div>
<div class="cap">Visión integral: volumen semanal, indicadores de sentimiento léxico (PySpark), cuota de mercados y series temporales comparadas.</div>
<div class="g2"><div><h3>Métricas Clave de la Semana</h3><div class="kg">
<div class="kc"><h4>Total Noticias</h4><div class="v" style="color:#3b5bdb">__kpi_total_str__</div><p>100% cobertura 7 días</p></div>
<div class="kc"><h4>Tono Promedio</h4><div class="v" style="color:#2f9e6b">__kpi_tono_promedio__</div><p>Balance global semanal</p></div>
<div class="kc"><h4>Tono Negativo</h4><div class="v" style="color:#e8590c">__kpi_tono_neg__</div><p>__kpi_neg_count__ noticias críticas</p></div>
<div class="kc"><h4>Medio Líder</h4><div class="v" style="font-size:1.05em;margin:.5em 0 .2em">__kpi_medio_top__</div><p>__kpi_medio_vol__ notas (Tono: __kpi_medio_tono__)</p></div>
<div class="kc"><h4>País Principal</h4><div class="v" style="color:#3b5bdb">__kpi_pais_top__</div><p>__kpi_pais_pct__ de cuota</p></div>
<div class="kc"><h4>Polaridad Léxica</h4><div class="v" style="color:#3b5bdb">__polaridad_semana__</div><p>Carga subjetiva promedio</p></div>
<div class="kc"><h4>Deporte Más Positivo</h4><div class="v" style="color:#2f9e6b;font-size:1.05em;margin:.4em 0 .2em">__kpi_dep_pos_nombre__</div><p>Tono: __kpi_dep_pos_tono__</p></div>
<div class="kc"><h4>Deporte Más Crítico</h4><div class="v" style="color:#e8590c;font-size:1.05em;margin:.4em 0 .2em">__kpi_dep_neg_nombre__</div><p>Tono: __kpi_dep_neg_tono__</p></div>
<div class="kc"><h4>Día Pico</h4><div class="v" style="font-size:.95em;margin:.6em 0 .2em">__kpi_dia_pico__</div><p>Mayor actividad semanal</p></div>
</div></div>
<div><h3>Distribución Global por País</h3><div class="box">__html_g1__</div></div></div>
<div class="g2"><div><h3>Evolución Temporal Global</h3><div class="box">__html_g4__</div></div>
<div><h3>Evolución Diaria por Deporte</h3><div class="box">__html_g4_dep__</div></div></div>
</div>

<!-- VENTANA 2: DEPORTES POR PAÍS -->
<div class="sec">
<div class="sh"><b>Deportes por País y Matriz de Calor</b><span>Especialización deportiva regional</span></div>
<div class="cap">Porcentajes normalizados de cobertura que aporta cada país a cada disciplina, eliminando distorsiones por volumen bruto de mercado.</div>
<div class="g2"><div><h3>Matriz de Calor: % de Aporte por País</h3><div class="box">__html_g2__</div></div>
<div><h3>Deporte Dominante por Región</h3><div class="box">__html_mapa__</div></div></div>
</div>

<!-- VENTANA 3: MEDIOS CON TAPAS DE DIARIOS -->
<div class="sec">
<div class="sh"><b>Participación y Concentración de Medios</b><span>Ecosistema editorial hispanohablante</span></div>
<div class="cap">Volumen de los medios de mayor cobertura y el podio con las tapas y portadas impresas del Top 3 de la semana.</div>
<div class="g2"><div><h3>Medios de Mayor Cobertura</h3><div class="box">__html_g3__</div></div>
<div><h3>Top 3 Medios de la Semana (Portadas y Volumen)</h3><div class="podio">__html_podio__</div></div></div>
</div>

<!-- VENTANA 4: TONO Y POLARIZACIÓN (pyspark_tonos_por_deporte) -->
<div class="sec">
<div class="sh"><b>Tono Emocional y Polarización</b><span>Termómetro de sentimiento léxico (GDELT 2.0 V2Tone & PySpark)</span></div>
<div class="cap">Evaluación multidimensional de las noticias deportivas: polaridad léxica, densidad de términos laudatorios vs críticos y clasificación por disciplina.</div>

<div class="kg" style="margin-bottom:.8em">
<div class="kc"><h4>Tono Promedio Global</h4><div class="v" style="color:#2f9e6b">__kpi_tono_promedio__</div><p>Balance levemente optimista</p></div>
<div class="kc"><h4>Cobertura Favorable</h4><div class="v" style="color:#2f9e6b">__kpi_pct_positivo__</div><p>__kpi_pos_count__ notas (tono &gt; 0)</p></div>
<div class="kc"><h4>Cobertura Crítica</h4><div class="v" style="color:#e8590c">__kpi_tono_neg__</div><p>__kpi_neg_count__ notas (tono &lt; 0)</p></div>
<div class="kc"><h4>Polaridad Léxica</h4><div class="v" style="color:#3b5bdb">__kpi_polaridad_prom__</div><p>Intensidad afectiva promedio</p></div>
<div class="kc"><h4>Positividad Media</h4><div class="v" style="color:#2f9e6b">__kpi_tono_positivo__</div><p>Frecuencia de términos positivos</p></div>
<div class="kc"><h4>Negatividad Media</h4><div class="v" style="color:#e8590c">__kpi_tono_negativo__</div><p>Frecuencia de términos de fricción</p></div>
</div>

<div class="g2">
<div><h3>Tono Emocional Promedio por Deporte (pyspark_tonos_por_deporte)</h3><div class="box">__html_g5__</div></div>
<div>
  <h3>Disciplinas en los Extremos</h3>
  <div class="deps">__html_tono_cards__</div>
  <div class="insight-box">
    <h4>💡 Hallazgos Clave de Sentimiento Léxico</h4>
    <p>• <b>Disciplinas de fricción:</b> Deportes de contacto como <b>Rugby</b> (-0.46) y <b>Boxeo / MMA</b> (-0.15) registran mayor negatividad relativa debido a reportes sobre sanciones, lesiones y polémicas arbitrales.</p>
    <p>• <b>Disciplinas de celebración:</b> <b>Atletismo / Natación</b> (+1.66) y <b>Baloncesto</b> (+1.06) destacan con los picos más positivos gracias a la cobertura laudatoria de récords, victorias y distinciones individuales.</p>
    <p>• <b>Fútbol como eje tonal:</b> Con más del 52% del caudal informativo de la semana, mantiene un tono moderadamente positivo (+0.41) que estabiliza el promedio de la agenda en español.</p>
  </div>
</div>
</div>
</div>

</div>
<div class="ctl"><button id="__ID__-prev">◀ Anterior</button><button class="pr" id="__ID__-next">Siguiente ▶</button><span class="pos" id="__ID__-pos"></span><span class="hint">← → navegar · F pantalla completa</span></div>
<script>
(function(){
var ID="__ID__",root=document.getElementById(ID),i=0;
function $(k){return document.getElementById(ID+"-"+k)}
var secs=root.querySelectorAll(".sec"),tabs=root.querySelectorAll(".tabs button"),n=secs.length;
if(location.protocol=="blob:")root.classList.add("fs");
function show(k){i=Math.max(0,Math.min(n-1,k));
 secs.forEach(function(s,j){s.style.display=j==i?"block":"none"});
 tabs.forEach(function(b,j){b.className=j==i?"on":j<i?"done":""});
 $("bar").style.width=((i+1)/n*100)+"%";$("pos").textContent="Sección "+(i+1)+"/"+n;
 setTimeout(function(){window.dispatchEvent(new Event("resize"))},60)}
function fs(){
 var ok=function(){var b=new Blob(['<!doctype html><meta charset="utf-8"><body style="margin:0">'+root.outerHTML],{type:"text/html"});
  var w=window.open(URL.createObjectURL(b),"_blank");if(!w)alert("El navegador bloqueó la pestaña nueva.")};
 if(document.fullscreenElement){document.exitFullscreen();return}
 try{var r=root.requestFullscreen&&root.requestFullscreen();if(r&&r.catch)r.catch(ok);else if(!r)ok()}catch(e){ok()}}
tabs.forEach(function(b,j){b.onclick=function(){show(j)}});
$("next").onclick=function(){show(i+1)};$("prev").onclick=function(){show(i-1)};$("fs").onclick=fs;
root.addEventListener("keydown",function(e){var k=e.key;
 if(k=="ArrowRight")show(i+1);else if(k=="ArrowLeft")show(i-1);else if(k=="f"||k=="F")fs()});
root.addEventListener("click",function(){root.focus()});
show(0);
})();
</script>
</div>
'''

_vals = dict(
    kpi_total_str=kpi_total_str, kpi_var_dia=kpi_var_dia, kpi_tono_neg=kpi_tono_neg, kpi_medio_top=kpi_medio_top,
    kpi_medio_vol=kpi_medio_vol, kpi_medio_tono=kpi_medio_tono, kpi_pais_top=kpi_pais_top, kpi_pais_pct=kpi_pais_pct,
    polaridad_semana=polaridad_semana, kpi_deportes_count=kpi_deportes_count, kpi_medios_totales=kpi_medios_totales,
    kpi_dia_pico=kpi_dia_pico, kpi_tono_promedio=kpi_tono_promedio, kpi_tono_positivo=kpi_tono_positivo,
    kpi_tono_negativo=kpi_tono_negativo, kpi_polaridad_prom=kpi_polaridad_prom, kpi_pos_count=kpi_pos_count,
    kpi_pct_positivo=kpi_pct_positivo, kpi_neg_count=kpi_neg_count,
    kpi_dep_pos_nombre=kpi_dep_pos_nombre, kpi_dep_pos_tono=kpi_dep_pos_tono,
    kpi_dep_neg_nombre=kpi_dep_neg_nombre, kpi_dep_neg_tono=kpi_dep_neg_tono,
    html_g1=html_g1, html_g4=html_g4, html_g4_dep=html_g4_dep, html_g2=html_g2, html_g3=html_g3,
    html_g5=html_g5, html_mapa=html_mapa, html_podio=html_podio, html_tono_cards=html_tono_cards
)

_pagina = _html.replace('__ID__', 'gx' + uuid.uuid4().hex[:6])
for _k, _v in _vals.items():
    _pagina = _pagina.replace(f'__{_k}__', str(_v))

display(HTML(_pagina))

# Opcional: guardar como .html para abrirlo en el navegador (F11)
with open('dashboard_gdelt.html', 'w', encoding='utf-8') as f:
    f.write('<!doctype html><meta charset="utf-8"><body style="margin:0">' + _pagina)
